# EBYS belge yönlendirme deneyleri (Colab) — v5

v5: gömme sınıflandırıcı ızgarası 4 paralel işte (yoğun gömmelerde doğrusal SVM yok); BDM yanıtları için ikinci geçiş (yeniden sorgu).
v4: fine-tuning T4'te fp16 + dinamik dolgu ile ~10 kat hızlı; gömme sınıflandırıcıları fine-tuning ile aynı anda arka planda koşar.
Claude çağrıları ek defterde (EBYS_LLM_ek.ipynb) yapılır; bu defter Gemini ve GPT'yi çalıştırır.

**Nasıl çalıştırılır:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **GPU** (A100 > L4 > T4) ve varsa **Yüksek RAM**.
Sonra **Çalışma zamanı → Tümünü çalıştır**. Drive ve Secrets için çıkan izin pencerelerini onaylayın.

* Ham veri: Drive'da `Drive'ım/ebys-routing/data/` klasörüne `Rapor-Orjinal-HamIlkHali-18102024-v3.xlsx`
  (isteğe bağlı: `Rapor-Onislenmis.xlsx`). Dosya adını değiştirmeniz gerekmez.
* API anahtarları: Colab Secrets (sol menüde anahtar simgesi); "Not defteri erişimi" açık olmalı.
* Her ara sonuç Drive'a yazılır. Bağlantı koparsa yeniden bağlanıp **Tümünü çalıştır** deyin; biten işler atlanır.
* Sonunda `ebys_results_<tarih>.zip` otomatik iner (ayrıca Drive'da `ebys-routing/` klasörüne kopyalanır).

Akış: veri hazırlama → [arka planda: TF-IDF (CPU) ve BDM API çağrıları] → gömme çıkarımı (GPU) → fine-tuning (GPU)
→ gömme sınıflandırıcıları (CPU) → arka plan işlerini bekleme → analiz → zip + indirme.

In [ ]:
# === 1. Drive bağlantısı, klasörler ve veri dosyalarının bulunması ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, json, time, shutil, glob
BASE = "/content/drive/MyDrive/ebys-routing"
for d in ["data", "src", "paper", "results", "figures", "cache", "logs"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)
os.chdir(BASE)

import pandas as pd

def _candidates(patterns):
    out = []
    for folder in [f"{BASE}/data", BASE, "/content/drive/MyDrive"]:
        for pat in patterns:
            for h in sorted(glob.glob(os.path.join(folder, pat))):
                if h not in out:
                    out.append(h)
    return out

def _cols(path):
    try:
        return set(pd.read_excel(path, nrows=0).columns)
    except Exception:
        return set()

RAW = f"{BASE}/data/raw_dataset.xlsx"
if not os.path.exists(RAW):
    need = {"EvrakId", "EvrakKonusu", "EvrakIcerigi", "EvrakinGeregineKisiKullaniciBirimi", "EvrakinSevkedilmeTarihi"}
    for c in _candidates(["Rapor-Orjinal-HamIlkHali-18102024-v3.xlsx", "*HamIlkHali*v3.xlsx", "*HamIlkHali*.xlsx"]):
        cols = _cols(c)
        if need <= cols and not ({"English", "MessageOnislenmis"} & cols):     # the raw export, not a derived file
            shutil.copy(c, RAW); print("Ham veri bulundu:", c, "-> data/raw_dataset.xlsx"); break
V1 = f"{BASE}/data/preprocessed_dataset_v1.xlsx"
if not os.path.exists(V1):
    for c in _candidates(["Rapor-Onislenmis.xlsx", "*Onislenmis*.xlsx", "*Önişlenmiş*.xlsx"]):
        if {"EvrakId", "English"} <= _cols(c):
            shutil.copy(c, V1); print("İngilizce çeviri sütunlu dosya bulundu:", c, "-> data/preprocessed_dataset_v1.xlsx"); break
if not os.path.exists(RAW):
    raise FileNotFoundError("Ham veri bulunamadı. Rapor-Orjinal-HamIlkHali-18102024-v3.xlsx dosyasını Drive'da "
                            "'Drive'ım/ebys-routing/data' klasörüne yükleyip bu hücreyi yeniden çalıştırın.")
print("Çalışma klasörü:", BASE)
print("data/:", os.listdir(f"{BASE}/data"))
if not os.path.exists(V1):
    print("Not: Rapor-Onislenmis.xlsx yok; yalnızca çeviri ablasyonu atlanacak, diğer her şey çalışır.")

In [ ]:
# === 2. Yapılandırma, donanım kontrolü ve API anahtarları ===
RUN_TFIDF, RUN_EMBED, RUN_FINETUNE, RUN_LLM, RUN_ANALYSIS = True, True, True, True, True

EMBED_MODELS    = ["minilm_multi", "distiluse_multi", "sbert_tr", "berturk_mean", "mpnet_multi", "e5_base", "labse",
                   "e5_large", "bge_m3", "qwen3_0_6b", "qwen3_4b"]
FINETUNE_MODELS = ["dbmdz/bert-base-turkish-cased", "xlm-roberta-base"]
LLM_MODELS      = ["gemini-3.8-flash", "gpt-6-luna"]     # Claude ek defterde (EBYS_LLM_ek.ipynb) çalıştırılıyor
SUMMARY_MODELS  = ["claude-haiku-5-5", "claude-sonnet-5-5", "gpt-6-luna", "gemini-3.8-flash"]   # özetlenecek tüm BDM'ler
LLM_MODES       = ["zeroshot", "rag10"]   # zeroshot -> yalnız test; rag10 -> test + doğrulama (kademe eşiği için)
LLM_WORKERS     = 6                       # sağlayıcı başına eşzamanlı istek; 429 hataları görürseniz 3'e indirin

import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU bulunamadı. Çalışma zamanı -> Çalışma zamanı türünü değiştir -> GPU (A100/L4/T4) seçip "
                       "Tümünü çalıştır'a yeniden basın.")
GPU = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print(f"GPU: {GPU} ({GPU_GB:.0f} GB) | RAM: {RAM_GB:.0f} GB | CPU çekirdeği: {os.cpu_count()}")
PARALLEL = RAM_GB >= 20     # yeterli RAM varsa TF-IDF, GPU işleriyle aynı anda arka planda koşar
if not PARALLEL:
    print("RAM 20 GB'ın altında: TF-IDF arka planda değil, GPU işlerinden sonra sırayla koşacak (daha uzun sürer). "
          "Mümkünse 'Yüksek RAM'i açın.")

from google.colab import userdata
CANDIDATES = {
    "ANTHROPIC_API_KEY": ["ANTHROPIC_API_KEY", "CLAUDE_API_KEY", "ANTHROPIC_KEY", "CLAUDE_KEY", "CLAUDE", "ANTHROPIC"],
    "OPENAI_API_KEY":    ["OPENAI_API_KEY", "OPENAI_KEY", "CHATGPT_API_KEY", "GPT_API_KEY", "OPENAI", "CHATGPT"],
    "GOOGLE_API_KEY":    ["GOOGLE_API_KEY", "GEMINI_API_KEY", "GEMINI_KEY", "GOOGLE_GENAI_API_KEY", "GEMINI", "GOOGLE_AI_KEY"],
}
found = {}
for env_name, names in CANDIDATES.items():
    for n in names:
        try:
            v = userdata.get(n)
        except Exception:
            v = None
        if v:
            os.environ[env_name] = v.strip(); found[env_name] = n; break
print("Bulunan anahtarlar:", found if found else "YOK")
def _provider(m):
    m = m.lower()
    return "ANTHROPIC_API_KEY" if m.startswith("claude") else "OPENAI_API_KEY" if m.startswith(("gpt", "o")) else "GOOGLE_API_KEY"
missing = [m for m in LLM_MODELS if _provider(m) not in found]
if missing:
    print("UYARI: anahtarı bulunamayan modeller atlanacak:", missing,
          "- Secrets'taki adlar şunlardan biri olmalı:", {k: v[:2] for k, v in CANDIDATES.items()})
LLM_MODELS = [m for m in LLM_MODELS if _provider(m) in found]
print("Çalıştırılacak BDM'ler:", LLM_MODELS)

In [ ]:
# === 3. Paketler (Colab'ın hazır numpy/pandas/torch sürümlerine dokunulmaz) ===
%pip install -q zeyrek snowballstemmer
%pip install -q anthropic openai google-genai
%pip install -q sentence-transformers statsmodels openpyxl
import nltk; nltk.download("punkt_tab", quiet=True)
!apt-get -qq install -y graphviz > /dev/null 2>&1 || true
import platform, datetime, importlib.metadata as im
def _v(p):
    try: return im.version(p)
    except Exception: return None
env = {"python": platform.python_version(), "sklearn": _v("scikit-learn"), "torch": _v("torch"), "transformers": _v("transformers"),
       "sentence_transformers": _v("sentence-transformers"), "zeyrek": _v("zeyrek"), "anthropic": _v("anthropic"), "openai": _v("openai"),
       "google_genai": _v("google-genai"), "gpu": GPU, "gpu_gb": round(GPU_GB), "ram_gb": round(RAM_GB), "cpu_count": os.cpu_count(),
       "date": datetime.date.today().strftime("%B %Y")}
json.dump(env, open(f"{BASE}/results/environment.json", "w"), indent=2); print(env)

## Kaynak kodu (src/)
Aşağıdaki hücreler pipeline modüllerini Drive'daki `src/` klasörüne yazar (yerel çalışmayla birebir aynı kod).

In [ ]:
%%writefile /content/drive/MyDrive/ebys-routing/src/prepare_data.py
"""
prepare_data.py
---------------
Builds the document-level routing dataset from the raw EDMS dispatch log and
defines the evaluation protocol (chronological split + stratified group folds).

Input : data/raw_dataset.xlsx            (one row per dispatch record)
        data/preprocessed_dataset_v1.xlsx (optional; supplies the machine-translated
                                           English content from the first study)
Output: data/documents.pkl               (one row per document)
        results/dataset_stats.json       (all counts reported in the paper)

Dataset construction (every step is counted and written to dataset_stats.json):
  1. Keep dispatch records that have a target unit (EvrakinGeregineKisiKullaniciBirimi).
  2. Text = subject + ". " + content.  Drop records whose text is empty.
  3. Broadcast filter: identical texts dispatched to more than BROADCAST_MAX distinct
     units are circulars/announcements, not routing decisions -> removed.
  4. One row per document (EvrakId). The label is the target unit of the FIRST dispatch
     (earliest EvrakinSevkedilmeTarihi).  The set of all units that received the
     document for action is kept as `label_set` for lenient evaluation.
  5. Near-identical documents are grouped by a normalised text key (`group_id`) so
     that identical texts never fall on both sides of a split.

Splits:
  * chronological (deployment-realistic): train = documents dated <= 2021-12-31,
    val = 2022-01-01 .. 2022-06-30, test = 2022-07-01 .. 2022-12-31.
  * 5-fold StratifiedGroupKFold (two repeats with different seeds) over all documents,
    groups = group_id, for variance estimates.
"""
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

ROOT = Path(__file__).resolve().parents[1]
RAW = ROOT / "data" / "raw_dataset.xlsx"
PRE_V1 = ROOT / "data" / "preprocessed_dataset_v1.xlsx"
OUT = ROOT / "data" / "documents.pkl"
STATS = ROOT / "results" / "dataset_stats.json"

LABEL = "EvrakinGeregineKisiKullaniciBirimi"
SENDER = "EvrakiSevkedenKullaniciBirimi"
BROADCAST_MAX = 3          # identical text dispatched to > 3 distinct units = broadcast
VAL_START, TEST_START, TEST_END = "2022-01-01", "2022-07-01", "2023-01-01"
SEEDS = (0, 1)             # repeats of the 5-fold grouped CV

# organisational hierarchy (parent department of each unit)
PARENT = {
    "Bilgi İşlem Daire Başkanlığı": "BİDB",
    "Yazılım Şube Müdürlüğü": "BİDB",
    "Bilgi İşlem Şube Müdürlüğü": "BİDB",
    "İnternet Hizmetleri Şube Müdürlüğü": "BİDB",
    "Haberleşme Hizmetleri Şube Müdürlüğü": "BİDB",
    "Sağlık Kültür ve Spor Daire Başkanlığı": "SKS",
    "Kültür Hizmetleri Şube Müdürlüğü": "SKS",
    "Satın Alma Şube Müdürlüğü": "SKS",
    "Sosyal Tesisler Şube Müdürlüğü": "SKS",
    "Beslenme Hizmetleri Şube Müdürlüğü": "SKS",
    "Spor Tesisleri Şube Müdürlüğü": "SKS",
    "Sağlık Hizmetleri Şube Müdürlüğü": "SKS",
    "Rektörlük": "Rektörlük",
}


def tr_lower(s: str) -> str:
    return s.replace("İ", "i").replace("I", "ı").lower()


def norm_key(s: str) -> str:
    """Normalised key used to group identical/near-identical texts."""
    s = tr_lower(str(s))
    s = re.sub(r"[^a-zçğıöşü]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def main():
    stats = {}
    df = pd.read_excel(RAW)
    stats["n_dispatch_records_raw"] = int(len(df))
    stats["n_documents_raw"] = int(df["EvrakId"].nunique())
    stats["date_min_raw"] = str(df["EvrakTarihi"].min().date())
    stats["date_max_raw"] = str(df["EvrakTarihi"].max().date())

    # 1. target unit present
    df = df[df[LABEL].notna()].copy()
    stats["n_records_with_target_unit"] = int(len(df))

    # 2. text
    df["subject"] = df["EvrakKonusu"].fillna("").astype(str).str.strip()
    df["content"] = df["EvrakIcerigi"].fillna("").astype(str).str.strip()
    df["text"] = (df["subject"] + ". " + df["content"]).str.strip(" .")
    df = df[df["text"].str.len() > 0].copy()
    stats["n_records_nonempty_text"] = int(len(df))
    stats["n_records_empty_content"] = int((df["content"] == "").sum())

    # 3. broadcast filter
    units_per_text = df.groupby("text")[LABEL].nunique()
    broadcast_texts = units_per_text[units_per_text > BROADCAST_MAX].index
    is_bc = df["text"].isin(broadcast_texts)
    stats["broadcast_threshold_units"] = BROADCAST_MAX
    stats["n_broadcast_texts"] = int(len(broadcast_texts))
    stats["n_broadcast_records_removed"] = int(is_bc.sum())
    stats["n_documents_with_target_unit"] = int(df["EvrakId"].nunique())
    bc_docs = df.loc[is_bc].drop_duplicates("EvrakId")
    stats["n_broadcast_documents_removed"] = int(df["EvrakId"].nunique() - df.loc[~is_bc, "EvrakId"].nunique())
    stats["n_broadcast_documents_empty_body"] = int((bc_docs["content"] == "").sum())
    stats["n_broadcast_documents_bare_petition"] = int(((bc_docs["content"] == "") & (bc_docs["subject"].str.upper() == "DİLEKÇE")).sum())
    stats["n_broadcast_documents_with_body"] = int((bc_docs["content"] != "").sum())
    stats["broadcast_units_per_text_median"] = float(units_per_text[broadcast_texts].median())
    df = df[~is_bc].copy()
    stats["n_records_after_broadcast_filter"] = int(len(df))

    # 4. one row per document: first dispatch defines the label
    df["dispatch_time"] = pd.to_datetime(df["EvrakinSevkedilmeTarihi"], errors="coerce", format="mixed")
    df = df.sort_values(["EvrakId", "dispatch_time"], kind="stable")
    g = df.groupby("EvrakId", sort=False)
    docs = g.first()[["EvrakTarihi", "subject", "content", "text", SENDER, LABEL, "dispatch_time",
                      "EvrakiSevkedenKullaniciUnvani", "EvrakKategorisiAdi"]].copy()
    docs = docs.rename(columns={"EvrakTarihi": "date", SENDER: "sender_unit", LABEL: "label",
                                "EvrakiSevkedenKullaniciUnvani": "sender_title",
                                "EvrakKategorisiAdi": "category"})
    docs["label_set"] = g[LABEL].agg(lambda s: tuple(dict.fromkeys(s)))   # ordered unique
    docs["n_dispatches"] = g.size()
    docs["n_units"] = docs["label_set"].map(len)
    docs = docs.reset_index().rename(columns={"EvrakId": "doc_id"})
    docs["date"] = pd.to_datetime(docs["date"])
    docs["department"] = docs["label"].map(PARENT)
    docs["sender_department"] = docs["sender_unit"].map(PARENT)
    stats["n_documents"] = int(len(docs))
    stats["n_documents_multi_unit"] = int((docs["n_units"] > 1).sum())
    stats["n_documents_self_dispatch"] = int((docs["sender_unit"] == docs["label"]).sum())
    stats["n_classes"] = int(docs["label"].nunique())
    stats["class_counts"] = docs["label"].value_counts().to_dict()
    stats["sender_unit_counts"] = docs["sender_unit"].value_counts().to_dict()
    stats["date_min"] = str(docs["date"].min().date())
    stats["date_max"] = str(docs["date"].max().date())
    stats["docs_per_year"] = docs["date"].dt.year.value_counts().sort_index().to_dict()
    wc = docs["text"].str.split().str.len()
    stats["words_mean"], stats["words_median"], stats["words_p90"], stats["words_max"] = float(wc.mean()), float(wc.median()), float(wc.quantile(.9)), int(wc.max())
    sc = docs["subject"].str.split().str.len()
    stats["subject_words_mean"] = float(sc.mean())

    # 5. groups of identical texts
    docs["group_id"] = docs["text"].map(norm_key).factorize()[0]
    stats["n_unique_texts"] = int(docs["group_id"].nunique())
    dup_mask = docs.groupby("group_id")["doc_id"].transform("size") > 1
    stats["n_documents_with_identical_twin"] = int(dup_mask.sum())
    maj = docs.groupby("group_id")["label"].agg(lambda s: s.value_counts().iloc[0]).sum()
    stats["label_consistency_ceiling"] = float(maj / len(docs))   # max accuracy of any function of the text

    # chronological split
    docs["split"] = np.where(docs["date"] < VAL_START, "train",
                     np.where(docs["date"] < TEST_START, "val", "test"))
    assert (docs["date"] < TEST_END).all()
    stats["split_sizes"] = docs["split"].value_counts().to_dict()
    stats["split_class_counts"] = {s: docs.loc[docs.split == s, "label"].value_counts().to_dict() for s in ["train", "val", "test"]}
    # identical texts across chronological splits (reported, not removed: realistic for templated correspondence)
    tr_groups = set(docs.loc[docs.split == "train", "group_id"])
    stats["n_test_docs_with_identical_text_in_train"] = int(docs.loc[docs.split == "test", "group_id"].isin(tr_groups).sum())

    # grouped stratified folds (two repeats)
    for seed in SEEDS:
        skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        fold = np.full(len(docs), -1)
        for k, (_, te) in enumerate(skf.split(docs, docs["label"], docs["group_id"])):
            fold[te] = k
        docs[f"fold_s{seed}"] = fold

    # machine-translated English content from the first study (content only; subjects were not translated)
    docs["content_en"] = np.nan
    if PRE_V1.exists():
        try:
            v1 = pd.read_excel(PRE_V1, usecols=["EvrakId", "English"]).drop_duplicates("EvrakId")
            docs = docs.drop(columns="content_en").merge(v1.rename(columns={"EvrakId": "doc_id", "English": "content_en"}), on="doc_id", how="left")
            stats["n_documents_with_english_content"] = int(docs["content_en"].notna().sum())
        except Exception as e:
            print("WARNING: could not read the English column from", PRE_V1.name, "-", e, "(translation ablation will be skipped)")

    docs.to_pickle(OUT)
    STATS.parent.mkdir(exist_ok=True)
    STATS.write_text(json.dumps(stats, ensure_ascii=False, indent=2, default=str))
    print(json.dumps(stats, ensure_ascii=False, indent=2, default=str))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/drive/MyDrive/ebys-routing/src/preprocess.py
"""
preprocess.py
-------------
Turkish-native preprocessing variants used by the lexical (TF-IDF) models.

  raw   : Turkish-aware lower-casing, letters only, stop words removed
  stem  : raw + Snowball Turkish stemmer (suffix stripping)
  lemma : raw + Zeyrek morphological analysis (lemma of the most likely parse)
  en    : machine-translated English content (from the first study; content only)

Contextual encoders receive the cleaned text with original casing and punctuation
(`text_clean`), because sub-word tokenisers are trained on natural text.
"""
import html
import json
import logging
import re
import sys
from functools import lru_cache
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
DOCS = ROOT / "data" / "documents.pkl"
OUT = ROOT / "data" / "text_variants.pkl"

# NLTK Turkish stop-word list (53 words) embedded so the pipeline has no download step
TR_STOPWORDS = set("""acaba ama aslında az bazı belki biri birkaç birşey biz bu çok çünkü da daha de defa diye eğer en
gibi hem hep hepsi her hiç için ile ise kez ki kim mı mu mü nasıl ne neden nerde nerede nereye niçin niye o sanki
şey siz şu tüm ve veya ya yani""".split())
# institutional boiler-plate that carries no routing information (closing formulas)
TR_STOPWORDS |= {"arz", "ederim", "rica", "gereğini", "bilgilerinizi", "bilgilerinize", "hk", "hakkında"}


def tr_lower(s: str) -> str:
    return s.replace("İ", "i").replace("I", "ı").lower()


def clean_text(s: str) -> str:
    """Light cleaning for contextual encoders: unescape HTML entities, normalise whitespace."""
    s = html.unescape(str(s))
    s = re.sub(r"<[^>]+>", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def tokens_raw(s: str):
    s = tr_lower(clean_text(s))
    s = re.sub(r"[^a-zçğıöşü]+", " ", s)
    return [w for w in s.split() if w not in TR_STOPWORDS and len(w) > 1]


_stemmer = None


def stem_word(w: str) -> str:
    global _stemmer
    if _stemmer is None:
        import snowballstemmer
        _stemmer = snowballstemmer.stemmer("turkish")
    return _stemmer.stemWord(w)


_analyzer = None


@lru_cache(maxsize=None)
def lemma_word(w: str) -> str:
    """Zeyrek lemma. Zeyrek has no contextual disambiguation, so among the candidate
    lemmas of a word the longest stem is taken (longest-stem heuristic; ties -> first
    analysis). Falls back to the surface form when no analysis exists."""
    global _analyzer
    if _analyzer is None:
        import zeyrek
        logging.getLogger("zeyrek").setLevel(logging.ERROR)
        _analyzer = zeyrek.MorphAnalyzer()
    try:
        res = _analyzer.lemmatize(w)
        cands = sorted(set(res[0][1])) if res else []
        if cands:
            return tr_lower(max(cands, key=len))      # longest stem; ties -> alphabetical (deterministic)
    except Exception:
        pass
    return w


def variant(text: str, kind: str) -> str:
    toks = tokens_raw(text)
    if kind == "raw":
        return " ".join(toks)
    if kind == "stem":
        return " ".join(stem_word(w) for w in toks)
    if kind == "lemma":
        return " ".join(lemma_word(w) for w in toks)
    raise ValueError(kind)


def ensure_nltk():
    """Zeyrek needs NLTK's punkt tokenizer; download it once if missing."""
    try:
        import nltk
        try:
            nltk.data.find("tokenizers/punkt_tab/turkish/")
        except LookupError:
            import os
            os.environ.setdefault("NLTK_ALLOW_PROXIED_URLOPEN", "1")
            nltk.download("punkt_tab", quiet=True)
    except Exception as e:
        print("nltk setup:", e)


def main():
    ensure_nltk()
    docs = pd.read_pickle(DOCS)
    out = pd.DataFrame({"doc_id": docs["doc_id"]})
    out["text_clean"] = docs["text"].map(clean_text)
    out["content_clean"] = docs["content"].map(clean_text)
    for kind in ["raw", "stem", "lemma"]:
        out[f"text_{kind}"] = docs["text"].map(lambda t: variant(t, kind))
        out[f"content_{kind}"] = docs["content"].map(lambda t: variant(t, kind))
        print(kind, "done; example:", out[f"text_{kind}"].iloc[2][:120])
    # English arm (content only): lower-case, letters only, English stop words removed
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    def en_tokens(s):
        if not isinstance(s, str):
            return None
        s = re.sub(r"[^a-z]+", " ", s.lower())
        return " ".join(w for w in s.split() if w not in ENGLISH_STOP_WORDS and len(w) > 1)
    out["content_en"] = docs["content_en"].map(en_tokens)
    out["content_en_clean"] = docs["content_en"].map(lambda t: clean_text(t) if isinstance(t, str) else "")
    out.to_pickle(OUT)
    vocab = {k: len(set(" ".join(out[f"text_{k}"]).split())) for k in ["raw", "stem", "lemma"]}
    print("vocabulary sizes:", vocab)
    (ROOT / "results" / "preprocessing_stats.json").write_text(json.dumps(vocab, indent=2))


if __name__ == "__main__":
    # Zeyrek's candidate analyses depend on Python's string-hash seed; fix it for reproducibility
    import os
    if os.environ.get("PYTHONHASHSEED") != "0":
        os.environ["PYTHONHASHSEED"] = "0"
        os.execv(sys.executable, [sys.executable] + sys.argv)
    main()


In [ ]:
%%writefile /content/drive/MyDrive/ebys-routing/src/common.py
"""
common.py
---------
Shared evaluation protocol for every representation (TF-IDF, frozen embeddings,
fine-tuned transformers, LLM predictions).

Protocol
  * Model selection: fit on `train`, select hyper-parameters on `val` (chronological
    validation split) by macro-F1.
  * Final estimate: refit the selected configuration on train+val, predict `test`
    (second half of 2022) exactly once.  Per-document predictions and class scores
    are stored so that top-k routing, confidence-based deferral, McNemar tests and
    error analyses can be computed afterwards without re-training.
  * Robustness: 5-fold StratifiedGroupKFold (groups = identical texts), two repeats,
    with the selected configuration -> mean +/- sd over 10 folds.
"""
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_recall_fscore_support)
from sklearn.naive_bayes import ComplementNB, GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier

ROOT = Path(__file__).resolve().parents[1]
RESULTS = ROOT / "results"
PREDS = RESULTS / "preds"
FT_RUNS = RESULTS / "ft_runs_v2"       # fine-tuning runs (v2: dynamic padding, hardware-appropriate precision)
SEEDS = (0, 1)

PARENT = {
    "Bilgi İşlem Daire Başkanlığı": "BİDB", "Yazılım Şube Müdürlüğü": "BİDB",
    "Bilgi İşlem Şube Müdürlüğü": "BİDB", "İnternet Hizmetleri Şube Müdürlüğü": "BİDB",
    "Haberleşme Hizmetleri Şube Müdürlüğü": "BİDB",
    "Sağlık Kültür ve Spor Daire Başkanlığı": "SKS", "Kültür Hizmetleri Şube Müdürlüğü": "SKS",
    "Satın Alma Şube Müdürlüğü": "SKS", "Sosyal Tesisler Şube Müdürlüğü": "SKS",
    "Beslenme Hizmetleri Şube Müdürlüğü": "SKS", "Spor Tesisleri Şube Müdürlüğü": "SKS",
    "Sağlık Hizmetleri Şube Müdürlüğü": "SKS", "Rektörlük": "Rektörlük",
}
SHORT = {
    "Bilgi İşlem Daire Başkanlığı": "BİDB (dept.)", "Yazılım Şube Müdürlüğü": "Software",
    "Bilgi İşlem Şube Müdürlüğü": "IT Services", "İnternet Hizmetleri Şube Müdürlüğü": "Internet Services",
    "Haberleşme Hizmetleri Şube Müdürlüğü": "Communications",
    "Sağlık Kültür ve Spor Daire Başkanlığı": "SKS (dept.)", "Kültür Hizmetleri Şube Müdürlüğü": "Cultural Services",
    "Satın Alma Şube Müdürlüğü": "Procurement", "Sosyal Tesisler Şube Müdürlüğü": "Social Facilities",
    "Beslenme Hizmetleri Şube Müdürlüğü": "Catering", "Spor Tesisleri Şube Müdürlüğü": "Sports Facilities",
    "Sağlık Hizmetleri Şube Müdürlüğü": "Health Services", "Rektörlük": "Rectorate",
}


# ----------------------------------------------------------------------------- data
def load_docs():
    docs = pd.read_pickle(ROOT / "data" / "documents.pkl")
    labels = sorted(docs["label"].unique())
    return docs, labels


def load_variants():
    return pd.read_pickle(ROOT / "data" / "text_variants.pkl")


def split_indices(docs):
    idx = {s: np.where(docs["split"].values == s)[0] for s in ("train", "val", "test")}
    idx["trainval"] = np.concatenate([idx["train"], idx["val"]])
    return idx


def cv_folds(docs):
    """List of (train_idx, test_idx) for 5-fold x 2 repeats grouped stratified CV."""
    folds = []
    for seed in SEEDS:
        f = docs[f"fold_s{seed}"].values
        for k in range(5):
            folds.append((np.where(f != k)[0], np.where(f == k)[0]))
    return folds


# ----------------------------------------------------------------------------- metrics
def topk_hits(scores, y_idx, k):
    top = np.argsort(-scores, axis=1)[:, :k]
    return (top == y_idx[:, None]).any(axis=1)


def metrics(y_true, y_pred, labels, scores=None, label_sets=None):
    """y_true/y_pred: label strings.  scores: (n, n_classes) in `labels` order."""
    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)[0],
        "macro_recall": precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)[1],
    }
    yt = np.asarray(y_true); yp = np.asarray(y_pred)
    dep_t = np.array([PARENT[x] for x in yt]); dep_p = np.array([PARENT[x] for x in yp])
    m["department_accuracy"] = float((dep_t == dep_p).mean())
    if scores is not None:
        y_idx = np.array([labels.index(x) for x in yt])
        m["top2_accuracy"] = float(topk_hits(scores, y_idx, 2).mean())
        m["top3_accuracy"] = float(topk_hits(scores, y_idx, 3).mean())
    if label_sets is not None:
        m["lenient_accuracy"] = float(np.mean([p in s for p, s in zip(yp, label_sets)]))
    return {k: float(v) for k, v in m.items()}


def per_class_table(y_true, y_pred, labels):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    return pd.DataFrame({"unit": labels, "precision": p, "recall": r, "f1": f, "support": s})


def bootstrap_ci(y_true, y_pred, labels, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    yt = np.asarray(y_true); yp = np.asarray(y_pred); n = len(yt)
    accs, f1s = [], []
    for _ in range(n_boot):
        b = rng.integers(0, n, n)
        accs.append(accuracy_score(yt[b], yp[b]))
        f1s.append(f1_score(yt[b], yp[b], labels=labels, average="macro", zero_division=0))
    return {"accuracy_ci": [float(np.percentile(accs, 2.5)), float(np.percentile(accs, 97.5))],
            "macro_f1_ci": [float(np.percentile(f1s, 2.5)), float(np.percentile(f1s, 97.5))]}


def mcnemar_test(y_true, pred_a, pred_b):
    """Exact McNemar test on paired correctness (Dietterich, 1998)."""
    from statsmodels.stats.contingency_tables import mcnemar
    a = np.asarray(pred_a) == np.asarray(y_true); b = np.asarray(pred_b) == np.asarray(y_true)
    tbl = [[int((a & b).sum()), int((a & ~b).sum())], [int((~a & b).sum()), int((~a & ~b).sum())]]
    res = mcnemar(tbl, exact=True)
    return {"n_a_only": tbl[0][1], "n_b_only": tbl[1][0], "statistic": float(res.statistic), "p_value": float(res.pvalue)}


# ----------------------------------------------------------------------------- classifiers
def classifier_grid(dense):
    """Hyper-parameter grid per classifier family.  Dense features (embeddings) are
    standardised for the linear / kernel models; sparse TF-IDF rows are already L2-normalised."""
    def lin(est):
        return make_pipeline(StandardScaler(), est) if dense else est
    grid = {
        "LR": [(dict(C=c, class_weight=cw), lin(LogisticRegression(C=c, class_weight=cw, max_iter=5000)))
               for c in (0.1, 1, 10) for cw in (None, "balanced")],
        "LinearSVM": [(dict(C=c, class_weight=cw), lin(LinearSVC(C=c, class_weight=cw, max_iter=20000)))
                      for c in (0.1, 0.5, 1, 5) for cw in (None, "balanced")],
        "KNN": [(dict(k=k, metric="cosine", weights="distance"), KNeighborsClassifier(n_neighbors=k, metric="cosine", weights="distance"))
                for k in (1, 5, 15)],
        "RF": [(dict(n_estimators=200, max_features="sqrt", min_samples_leaf=l, class_weight=cw),
                RandomForestClassifier(n_estimators=200, max_features="sqrt", min_samples_leaf=l, class_weight=cw, n_jobs=-1, random_state=0))
               for l in (1, 3) for cw in (None, "balanced_subsample")],
        "DT": [(dict(min_samples_leaf=l), DecisionTreeClassifier(min_samples_leaf=l, random_state=0)) for l in (1, 5)],
    }
    if dense:
        # linear SVM on sparse TF-IDF only: on standardised dense embeddings liblinear needed up to
        # ~20 min per fit (C=5) and never beat LR or the RBF-SVM on the first two encoders (pilot)
        grid.pop("LinearSVM")
        # the RBF kernel is evaluated on dense embeddings only; on sparse n-gram spaces the
        # linear kernel is the standard (and here much cheaper) choice
        grid["RBF-SVM"] = [(dict(C=c, gamma="scale", class_weight=cw), lin(SVC(C=c, gamma="scale", class_weight=cw)))
                           for c in (1, 10) for cw in (None, "balanced")]
        grid["NB"] = [(dict(kind="GaussianNB"), GaussianNB())]
    else:
        grid["NB"] = [(dict(kind="ComplementNB", alpha=a), ComplementNB(alpha=a)) for a in (0.1, 0.5, 1.0)]
    return grid


def class_scores(est, X, labels):
    """(n, n_classes) scores aligned with `labels` (probabilities when available)."""
    final = est.steps[-1][1] if hasattr(est, "steps") else est
    if hasattr(final, "predict_proba"):
        s = est.predict_proba(X)
    else:
        s = est.decision_function(X)
        if s.ndim == 1:
            s = np.stack([-s, s], axis=1)
    cls = list(final.classes_)
    out = np.full((X.shape[0], len(labels)), -1e9 if not hasattr(final, "predict_proba") else 0.0)
    for j, c in enumerate(cls):
        out[:, labels.index(c)] = s[:, j]
    return out


# ----------------------------------------------------------------------------- features
class PrecomputedFeatures:
    """Rows of a precomputed matrix (e.g. frozen embeddings); nothing is fitted."""
    def __init__(self, X):
        self.X = X
        self.n_features = X.shape[1]

    def fit(self, idx):
        return self

    def transform(self, idx):
        return self.X[idx]


class TfidfFeatures:
    """TF-IDF fitted on the training rows only (vocabulary and IDF never see held-out text)."""
    def __init__(self, texts, char=False, **kw):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.texts = np.asarray(texts, dtype=object)
        params = dict(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=300000) if char \
            else dict(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")
        params.update(kw)
        self.vec = TfidfVectorizer(**params)
        self.n_features = None

    def fit(self, idx):
        self.vec.fit(self.texts[idx])
        self.n_features = len(self.vec.vocabulary_)
        return self

    def transform(self, idx):
        return self.vec.transform(self.texts[idx])


class UnionFeatures:
    def __init__(self, *feats):
        self.feats = feats
        self.n_features = None

    def fit(self, idx):
        for f in self.feats:
            f.fit(idx)
        self.n_features = sum(f.n_features for f in self.feats)
        return self

    def transform(self, idx):
        return sparse.hstack([f.transform(idx) for f in self.feats]).tocsr()


# ----------------------------------------------------------------------------- protocol
def run_representation(name, feat, docs, labels, dense, families=None, log=print, cv=True, extra_meta=None, results_file="results_main.csv"):
    """Full protocol for one feature extractor `feat` (PrecomputedFeatures / TfidfFeatures).

    1. every classifier family: hyper-parameters selected on `val` (macro-F1), model trained on `train`
    2. every family: refit on train+val, one evaluation on `test` (predictions + class scores stored)
    3. the family with the best validation macro-F1 is the representation's *selected system*;
       the grouped 5x2 CV is run for that family only (variance estimate of the reported system)
    Returns the result rows (one per family); rows are also appended to results/results_main.csv."""
    PREDS.mkdir(parents=True, exist_ok=True)
    (RESULTS / "val_grid").mkdir(parents=True, exist_ok=True)
    (RESULTS / "preds_val").mkdir(parents=True, exist_ok=True)
    idx = split_indices(docs)
    y = docs["label"].values
    label_sets = docs["label_set"].values
    grid = classifier_grid(dense)
    if families:
        grid = {k: v for k, v in grid.items() if k in families}
    # resumable: a representation whose families are all in results_main.csv is skipped
    main_csv = RESULTS / results_file
    if main_csv.exists():
        old = pd.read_csv(main_csv)
        done = set(old.loc[old["representation"] == name, "family"])
        if all(k in done and (PREDS / f"{name}__{k}.pkl").exists() for k in grid):
            log(f"[{name}] already done - skipping")
            return []
    feat.fit(idx["train"]); Xtr, Xva = feat.transform(idx["train"]), feat.transform(idx["val"])
    feat.fit(idx["trainval"]); Xtv, Xte = feat.transform(idx["trainval"]), feat.transform(idx["test"])
    n_features = feat.n_features
    selected = {}
    # --- 1. selection on the chronological validation split
    for fam, configs in grid.items():
        best, val_rows = None, []
        for params, est in configs:
            e = clone(est)
            t0 = time.time(); e.fit(Xtr, y[idx["train"]]); fit_t = time.time() - t0
            yp = e.predict(Xva)
            m = metrics(y[idx["val"]], yp, labels)
            val_rows.append({"family": fam, **params, **m, "fit_seconds": fit_t})
            if best is None or m["macro_f1"] > best[0]:
                best = (m["macro_f1"], params, est, yp, class_scores(e, Xva, labels))
        selected[fam] = best
        pd.DataFrame(val_rows).to_csv(RESULTS / "val_grid" / f"{name}__{fam}.csv", index=False)
        pd.to_pickle({"doc_id": docs["doc_id"].values[idx["val"]], "y_true": y[idx["val"]], "y_pred": best[3], "scores": best[4],
                      "labels": labels, "params": best[1], "family": fam, "representation": name},
                     RESULTS / "preds_val" / f"{name}__{fam}.pkl")
    chosen = max(selected, key=lambda f: selected[f][0])
    # --- 2. final fit on train+val, single evaluation on test (every family) ; 3. CV for the chosen family
    results = []
    for fam, (vf1, params, est, _, _) in selected.items():
        e = clone(est)
        t0 = time.time(); e.fit(Xtv, y[idx["trainval"]]); fit_t = time.time() - t0
        t0 = time.time(); yp = e.predict(Xte); pred_t = (time.time() - t0) / len(idx["test"])
        sc = class_scores(e, Xte, labels)
        m = metrics(y[idx["test"]], yp, labels, scores=sc, label_sets=label_sets[idx["test"]])
        ci = bootstrap_ci(y[idx["test"]], yp, labels)
        row = {"representation": name, "family": fam, "params": json.dumps(params), "val_macro_f1": vf1, "selected": fam == chosen,
               **m, **ci, "fit_seconds_trainval": fit_t, "predict_ms_per_doc": pred_t * 1000, "n_features": int(n_features), **(extra_meta or {})}
        pd.to_pickle({"doc_id": docs["doc_id"].values[idx["test"]], "y_true": y[idx["test"]], "y_pred": yp, "scores": sc, "labels": labels,
                      "params": params, "family": fam, "representation": name}, PREDS / f"{name}__{fam}.pkl")
        if cv and fam == chosen:
            accs, f1s = [], []
            for tr, te in cv_folds(docs):
                feat.fit(tr); Xa, Xb = feat.transform(tr), feat.transform(te)
                e = clone(est); e.fit(Xa, y[tr]); p = e.predict(Xb)
                accs.append(accuracy_score(y[te], p)); f1s.append(f1_score(y[te], p, labels=labels, average="macro", zero_division=0))
            row.update({"cv_accuracy_mean": float(np.mean(accs)), "cv_accuracy_sd": float(np.std(accs, ddof=1)),
                        "cv_macro_f1_mean": float(np.mean(f1s)), "cv_macro_f1_sd": float(np.std(f1s, ddof=1)),
                        "cv_folds": json.dumps({"acc": accs, "macro_f1": f1s})})
        results.append(row)
        append_results([row], file=results_file)   # written immediately so an interrupted run resumes
        log(f"[{name}] {fam:10s} {'*' if fam == chosen else ' '} {params}  val mF1={vf1:.4f} | test acc={m['accuracy']:.4f} macroF1={m['macro_f1']:.4f} wF1={m['weighted_f1']:.4f}"
            + (f"  CV acc={row['cv_accuracy_mean']:.4f}±{row['cv_accuracy_sd']:.4f}" if "cv_accuracy_mean" in row else ""))
    return results


def append_results(rows, file="results_main.csv"):
    """Upserts result rows (key: representation + family).  Every stage writes its own file
    (results_tfidf.csv, results_emb.csv, results_ft.csv, results_llm.csv) so that stages can run
    in parallel processes; the file is replaced atomically (write to a temp file, then rename)."""
    if not rows:
        return None
    f = RESULTS / file
    df = pd.DataFrame(rows)
    if f.exists():
        old = pd.read_csv(f)
        old = old[~(old["representation"] + "__" + old["family"]).isin(df["representation"] + "__" + df["family"])]
        df = pd.concat([old, df], ignore_index=True)
    tmp = f.with_suffix(".tmp.csv")
    df.to_csv(tmp, index=False)
    tmp.replace(f)
    return df


def load_all_results():
    """All result files merged (later files win on duplicate representation + family)."""
    files = sorted(RESULTS.glob("results_*.csv"))
    files = [f for f in files if not f.name.endswith(".tmp.csv")]
    if not files:
        return pd.DataFrame()
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return df.drop_duplicates(subset=["representation", "family"], keep="last").reset_index(drop=True)


In [ ]:
%%writefile /content/drive/MyDrive/ebys-routing/src/embed.py
"""
embed.py
--------
Extracts frozen sentence embeddings for every document and caches them as .npy files
(cache/emb_<model_key>__<text_field>.npy).  Re-running skips cached files, so the script
is resumable.  Usage:

    python src/embed.py                      # all CPU-feasible models on text_clean
    python src/embed.py --models e5_base labse --field text_clean
    python src/embed.py --list

Models are referenced by a short key; `MODELS` records the Hugging Face checkpoint,
the pooling strategy and the prefix required by instruction-tuned encoders.
"""
import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT = Path(__file__).resolve().parents[1]
CACHE = ROOT / "cache"
MAX_SEQ = 256

MODELS = {
    # key: (checkpoint, family, pooling, query prefix, notes)
    "minilm_multi": ("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "SBERT multilingual (distilled)", "mean", "", "118M params, 384-d"),
    "mpnet_multi": ("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", "SBERT multilingual", "mean", "", "278M params, 768-d"),
    "distiluse_multi": ("sentence-transformers/distiluse-base-multilingual-cased-v2", "multilingual USE (distilled)", "mean+dense", "", "135M params, 512-d"),
    "e5_base": ("intfloat/multilingual-e5-base", "mE5", "mean", "query: ", "278M params, 768-d"),
    "e5_large": ("intfloat/multilingual-e5-large", "mE5", "mean", "query: ", "560M params, 1024-d"),
    "labse": ("sentence-transformers/LaBSE", "LaBSE", "cls+dense", "", "471M params, 768-d"),
    "sbert_tr": ("emrecan/bert-base-turkish-cased-mean-nli-stsb-tr", "Turkish SBERT (BERTurk + NLI/STS)", "mean", "", "111M params, 768-d"),
    "berturk_mean": ("dbmdz/bert-base-turkish-cased", "BERTurk (no sentence fine-tuning)", "mean", "", "111M params, 768-d"),
    "bge_m3": ("BAAI/bge-m3", "BGE-M3", "cls", "", "568M params, 1024-d"),
    "mpnet_en": ("sentence-transformers/all-mpnet-base-v2", "SBERT English", "mean", "", "110M params, 768-d; English text only"),
    "qwen3_0_6b": ("Qwen/Qwen3-Embedding-0.6B", "Qwen3-Embedding", "last-token", "Instruct: Given a Turkish institutional document, identify the organisational unit responsible for it\nQuery: ", "0.6B params, 1024-d"),
    "qwen3_4b": ("Qwen/Qwen3-Embedding-4B", "Qwen3-Embedding", "last-token", "Instruct: Given a Turkish institutional document, identify the organisational unit responsible for it\nQuery: ", "4B params, 2560-d; GPU"),
}
GPU_EXTRA = ["e5_large", "bge_m3", "qwen3_0_6b", "qwen3_4b"]
CPU_DEFAULT = ["minilm_multi", "distiluse_multi", "sbert_tr", "berturk_mean", "mpnet_multi", "e5_base", "labse"]


def device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def load_encoder(key):
    from sentence_transformers import SentenceTransformer, models
    ckpt, _, pooling, _, _ = MODELS[key]
    kw = {}
    if device() == "cuda" and key.startswith("qwen3"):
        import transformers
        dkey = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"
        half = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16   # T4 only emulates bf16
        kw = {"model_kwargs": {dkey: half}, "tokenizer_kwargs": {"padding_side": "left"}}
    if key == "berturk_mean":
        word = models.Transformer(ckpt, max_seq_length=MAX_SEQ)
        pool = models.Pooling(word.get_word_embedding_dimension(), pooling_mode="mean")
        enc = SentenceTransformer(modules=[word, pool], device=device())
    else:
        enc = SentenceTransformer(ckpt, device=device(), **kw)
        enc.max_seq_length = MAX_SEQ
    return enc


def embed(key, texts, batch_size=None, log=print):
    ckpt, _, _, prefix, _ = MODELS[key]
    enc = load_encoder(key)
    batch_size = batch_size or (64 if device() == "cuda" else 32)
    t0 = time.time()
    X = enc.encode([prefix + t for t in texts], batch_size=batch_size, show_progress_bar=False,
                   convert_to_numpy=True, normalize_embeddings=False)
    dt = time.time() - t0
    if not np.isfinite(X).all():
        raise ValueError(f"{key}: {int((~np.isfinite(X)).any(axis=1).sum())} documents have non-finite embeddings "
                         "(fp16 overflow?) - not cached; run this model on a GPU with bf16 support (L4/A100)")
    log(f"{key}: {len(texts)} docs in {dt/60:.1f} min ({len(texts)/dt:.1f} docs/s), dim={X.shape[1]}")
    return X.astype(np.float32), dt


def main():
    global MAX_SEQ
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="*", default=None, help="default: CPU set, plus the large models when a GPU is present")
    ap.add_argument("--field", default="text_clean")
    ap.add_argument("--list", action="store_true")
    ap.add_argument("--max_seq", type=int, default=MAX_SEQ)
    args = ap.parse_args()
    if args.list:
        for k, v in MODELS.items():
            print(f"{k:16s} {v[0]:60s} {v[4]}")
        return
    MAX_SEQ = args.max_seq
    if args.models is None:
        args.models = CPU_DEFAULT + (GPU_EXTRA if torch.cuda.is_available() else [])
    torch.set_num_threads(max(1, torch.get_num_threads()))
    var = pd.read_pickle(ROOT / "data" / "text_variants.pkl")
    texts = var[args.field].fillna("").tolist()
    CACHE.mkdir(exist_ok=True)
    timing_file = CACHE / "embedding_times.json"
    timing = json.loads(timing_file.read_text()) if timing_file.exists() else {}
    failed = []
    for key in args.models:
        out = CACHE / f"emb_{key}__{args.field}.npy"
        if out.exists():
            print("cached:", out.name)
            continue
        try:
            X, dt = embed(key, texts)
        except Exception as e:          # one failing model must not stop the others
            print(f"!! FAILED {key}: {repr(e)[:400]}", flush=True)
            failed.append(key)
            import gc; gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue
        np.save(out, X)
        timing[f"{key}__{args.field}"] = {"seconds": dt, "docs": len(texts), "dim": int(X.shape[1]), "max_seq": MAX_SEQ,
                                          "device": device(), "threads": torch.get_num_threads()}
        timing_file.write_text(json.dumps(timing, indent=2))
        del X
        import gc; gc.collect()
        if device() == "cuda":
            torch.cuda.empty_cache()
    if failed:
        print("\nEmbedding extraction failed for:", failed, "(the other models are cached; re-run to retry)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/drive/MyDrive/ebys-routing/src/run_tfidf.py
"""
run_tfidf.py
------------
Lexical baselines under the shared protocol (see common.py).

Representations
  tfidf_word_raw     word 1-2 grams on raw Turkish tokens (lower-cased, stop words removed)
  tfidf_word_stem    ... on Snowball-stemmed tokens
  tfidf_word_lemma   ... on Zeyrek-lemmatised tokens
  tfidf_char_raw     character 2-5 grams (within word boundaries) on raw tokens
  tfidf_wordchar_raw union of word and character n-grams
Translation ablation (content only, identical documents in both arms)
  tfidf_content_tr   Turkish content, raw tokens
  tfidf_content_en   machine-translated English content (Google Translate, first study)
Input ablation
  tfidf_subject_raw  subject line only
"""
import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import numpy as np
from scipy import sparse

from common import (TfidfFeatures, UnionFeatures, append_results, load_docs, load_variants, run_representation)


class SenderAugmented:
    """Appends a one-hot encoding of the dispatching (sender) unit to a sparse text representation."""
    def __init__(self, feat, docs):
        self.feat = feat
        units = sorted(docs["sender_unit"].dropna().unique())
        M = np.zeros((len(docs), len(units)), dtype=np.float32)
        for j, u in enumerate(units):
            M[:, j] = (docs["sender_unit"].values == u)
        self.M = sparse.csr_matrix(M)
        self.n_features = None

    def fit(self, idx):
        self.feat.fit(idx); self.n_features = self.feat.n_features + self.M.shape[1]; return self

    def transform(self, idx):
        return sparse.hstack([self.feat.transform(idx), self.M[idx]]).tocsr()

REPS = {
    "tfidf_word_raw": lambda v: TfidfFeatures(v["text_raw"]),
    "tfidf_word_stem": lambda v: TfidfFeatures(v["text_stem"]),
    "tfidf_word_lemma": lambda v: TfidfFeatures(v["text_lemma"]),
    "tfidf_char_raw": lambda v: TfidfFeatures(v["text_raw"], char=True),
    "tfidf_wordchar_raw": lambda v: UnionFeatures(TfidfFeatures(v["text_raw"]), TfidfFeatures(v["text_raw"], char=True)),
    "tfidf_content_tr": lambda v: TfidfFeatures(v["content_raw"].fillna("")),
    "tfidf_content_en": lambda v: TfidfFeatures(v["content_en"].fillna("")),
    "tfidf_subject_raw": lambda v: TfidfFeatures(v["text_raw"].str.split(" ").str[:0].str.join(" ")),  # placeholder, replaced below
}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--reps", nargs="*", default=list(REPS))
    ap.add_argument("--families", nargs="*", default=None)
    ap.add_argument("--no_cv", action="store_true")
    ap.add_argument("--with_sender", action="store_true", help="append the sender-unit one-hot (context-aware variant)")
    args = ap.parse_args()
    docs, labels = load_docs()
    var = load_variants()
    # subject-only variant uses the raw tokens of the subject line
    from preprocess import variant
    var["subject_raw"] = docs["subject"].map(lambda t: variant(t, "raw"))
    REPS["tfidf_subject_raw"] = lambda v: TfidfFeatures(v["subject_raw"])
    for name in args.reps:
        feat = REPS[name](var)
        if args.with_sender:
            feat = SenderAugmented(feat, docs); name += "+sender"
        run_representation(name, feat, docs, labels, dense=False, families=args.families, cv=not args.no_cv,
                           results_file="results_tfidf.csv")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/drive/MyDrive/ebys-routing/src/run_embeddings.py
"""
run_embeddings.py
-----------------
Frozen sentence-embedding representations under the shared protocol.  Embeddings are
read from cache/emb_<key>__<field>.npy (see embed.py); each is evaluated with the full
classifier grid (dense=True).

    python src/run_embeddings.py                       # every cached embedding on text_clean
    python src/run_embeddings.py --keys e5_base labse
    python src/run_embeddings.py --keys e5_base --field content_clean   # input ablation
    python src/run_embeddings.py --keys e5_base --with_sender           # context-aware variant
"""
import argparse
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import PrecomputedFeatures, load_docs, run_representation
from embed import CACHE, MODELS


def sender_onehot(docs):
    units = sorted(docs["sender_unit"].dropna().unique())
    M = np.zeros((len(docs), len(units)), dtype=np.float32)
    for j, u in enumerate(units):
        M[:, j] = (docs["sender_unit"].values == u)
    return M


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--keys", nargs="*", default=None)
    ap.add_argument("--field", default="text_clean")
    ap.add_argument("--families", nargs="*", default=None)
    ap.add_argument("--with_sender", action="store_true")
    ap.add_argument("--no_cv", action="store_true")
    ap.add_argument("--results_file", default="results_emb.csv", help="one file per parallel job avoids write races")
    args = ap.parse_args()
    docs, labels = load_docs()
    keys = args.keys or [k for k in MODELS if (CACHE / f"emb_{k}__{args.field}.npy").exists()]
    for key in keys:
        f = CACHE / f"emb_{key}__{args.field}.npy"
        if not f.exists():
            print("missing embedding cache:", f.name, "- run embed.py first"); continue
        X = np.load(f)
        name = f"emb_{key}" + ("" if args.field == "text_clean" else f"__{args.field}")
        if args.with_sender:
            X = np.hstack([X, sender_onehot(docs)]); name += "+sender"
        meta = {"checkpoint": MODELS[key][0], "model_family": MODELS[key][1], "embedding_dim": int(X.shape[1])}
        if not np.isfinite(X).all():
            print(f"!! {name}: non-finite values in the cached embedding - skipped"); continue
        try:
            run_representation(name, PrecomputedFeatures(X), docs, labels, dense=True, families=args.families,
                               cv=not args.no_cv, extra_meta=meta, results_file=args.results_file)
        except Exception as e:
            print(f"!! FAILED {name}: {repr(e)[:300]}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/drive/MyDrive/ebys-routing/src/run_emb_parallel.py
"""
run_emb_parallel.py
-------------------
Runs the frozen-embedding classifier grid (run_embeddings.py) as several parallel jobs, one
representation per job, each writing its own results file (results_emb__<job>.csv) so that
parallel processes never write the same file.  Most scikit-learn fits here are single-threaded,
so N jobs with 2 BLAS threads each use the CPU far better than one sequential process.
Re-running resumes: finished representations are skipped by run_representation.

    python src/run_emb_parallel.py --workers 4
"""
import argparse
import os
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]

# longest first (embedding dimension), so that the large models do not finish last
JOBS = [
    ("qwen3_4b", "--keys qwen3_4b"),
    ("e5_large", "--keys e5_large"),
    ("bge_m3", "--keys bge_m3"),
    ("qwen3_0_6b", "--keys qwen3_0_6b"),
    ("sbert_tr", "--keys sbert_tr"),
    ("berturk_mean", "--keys berturk_mean"),
    ("mpnet_multi", "--keys mpnet_multi"),
    ("e5_base", "--keys e5_base"),
    ("labse", "--keys labse"),
    ("distiluse_multi", "--keys distiluse_multi"),
    ("minilm_multi", "--keys minilm_multi"),
    ("mpnet_multi__content_clean", "--keys mpnet_multi --field content_clean"),
    ("mpnet_en__content_en_clean", "--keys mpnet_en --field content_en_clean"),
    ("e5_base+sender", "--keys e5_base --with_sender"),
    ("mpnet_multi+sender", "--keys mpnet_multi --with_sender"),
]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--threads", type=int, default=2, help="BLAS / joblib threads per job")
    ap.add_argument("--only", nargs="*", default=None, help="run only these jobs (names as in JOBS)")
    a = ap.parse_args()
    jobs = [j for j in JOBS if a.only is None or j[0] in a.only]
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "OMP_NUM_THREADS": str(a.threads), "OPENBLAS_NUM_THREADS": str(a.threads),
           "MKL_NUM_THREADS": str(a.threads), "LOKY_MAX_CPU_COUNT": str(a.threads)}
    (ROOT / "logs").mkdir(exist_ok=True)
    t_start = time.time()
    status = {}

    def job(item):
        name, args = item
        t0 = time.time()
        print(time.strftime("%H:%M"), f"başladı  {name}", flush=True)
        cmd = f"{sys.executable} src/run_embeddings.py {args} --results_file results_emb__{name}.csv"
        with open(ROOT / "logs" / f"emb__{name}.log", "a") as log:
            log.write(f"\n===== {time.ctime()} :: {cmd}\n"); log.flush()
            rc = subprocess.run(cmd, shell=True, cwd=ROOT, env=env, stdout=log, stderr=subprocess.STDOUT).returncode
        status[name] = rc
        done = sum(1 for v in status.values() if v == 0)
        print(time.strftime("%H:%M"), f"bitti    {name} ({(time.time() - t0) / 60:.0f} dk, kod {rc}) - "
              f"{done}/{len(jobs)} tamam, toplam {(time.time() - t_start) / 60:.0f} dk", flush=True)
        return rc

    with ThreadPoolExecutor(max_workers=a.workers) as ex:
        list(ex.map(job, jobs))
    bad = [n for n, rc in status.items() if rc != 0]
    print(time.strftime("%H:%M"), "TÜM GÖMME İŞLERİ BİTTİ" + (f" - hata veren: {bad} (logs/emb__<ad>.log)" if bad else ""), flush=True)
    sys.exit(1 if bad else 0)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/drive/MyDrive/ebys-routing/src/finetune.py
"""
finetune.py
-----------
End-to-end fine-tuning of pre-trained encoders for 13-way routing (GPU recommended).

Protocol (same splits as every other model):
  1. selection : for each learning rate, train on `train` for up to MAX_EPOCHS and pick the
                 epoch with the best validation macro-F1 (seed 0)
  2. final     : retrain on train+val with the selected (lr, epochs) for SEEDS seeds and
                 predict `test` once per seed (per-document probabilities are stored)
  3. cv        : optional 5-fold grouped CV (seed-0 folds) with the selected configuration

Efficiency: sequences are truncated at MAX_LEN tokens but padded only to the longest sequence
of each batch, and training batches are formed from documents of similar length (length-grouped
sampling, as in Hugging Face `group_by_length`).  Mixed precision uses bf16 on GPUs with native
bf16 (compute capability >= 8.0: A100, L4, H100) and fp16 with loss scaling otherwise (T4, V100).

Every run writes results/ft_runs_v2/<tag>.json and is skipped when that file exists, so the
script resumes after an interruption.

    python src/finetune.py --cv --models dbmdz/bert-base-turkish-cased xlm-roberta-base
"""
import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset, Sampler

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import FT_RUNS, PREDS, RESULTS, append_results, bootstrap_ci, cv_folds, load_docs, load_variants, metrics, split_indices

FT_DIR = FT_RUNS
MAX_LEN, BATCH, EVAL_BATCH, MAX_EPOCHS, WARMUP, WD = 256, 32, 128, 6, 0.1, 0.01
SEEDS_FINAL = (0, 1, 2)


# ----------------------------------------------------------------------------- data
class DS(Dataset):
    """Holds unpadded token id lists; padding is done per batch by the collator."""
    def __init__(self, enc, ids, y):
        self.enc, self.ids, self.y = enc, np.asarray(ids), y

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        j = self.ids[i]
        d = {k: v[j] for k, v in self.enc.items()}
        d["labels"] = int(self.y[j])
        return d


class LengthGroupedBatchSampler(Sampler):
    """Shuffles documents, sorts them by length inside mega-batches of `mega` batches, cuts
    batches, and shuffles the batch order.  A new permutation is drawn every epoch from a
    seeded generator, so training is reproducible."""
    def __init__(self, lengths, batch_size, seed, mega=50):
        self.lengths = np.asarray(lengths); self.bs = batch_size; self.mega = mega
        self.rng = np.random.default_rng(seed)

    def __iter__(self):
        idx = self.rng.permutation(len(self.lengths))
        step = self.bs * self.mega
        batches = []
        for s in range(0, len(idx), step):
            chunk = idx[s:s + step]
            chunk = chunk[np.argsort(-self.lengths[chunk], kind="stable")]
            batches += [chunk[i:i + self.bs].tolist() for i in range(0, len(chunk), self.bs)]
        for b in self.rng.permutation(len(batches)):
            yield batches[b]

    def __len__(self):
        return (len(self.lengths) + self.bs - 1) // self.bs


def sorted_batches(lengths, batch_size):
    order = np.argsort(-np.asarray(lengths), kind="stable")
    return [order[i:i + batch_size].tolist() for i in range(0, len(order), batch_size)]


def tokenize(tok, texts):
    enc = tok(list(texts), truncation=True, max_length=MAX_LEN)          # no padding here
    return {k: list(v) for k, v in enc.items() if k in ("input_ids", "attention_mask", "token_type_ids")}


def amp_dtype(device):
    if device.type != "cuda":
        return torch.float32
    # is_bf16_supported() is also True on T4 through (slow) emulation, so check the hardware generation
    return torch.bfloat16 if torch.cuda.get_device_capability(device)[0] >= 8 else torch.float16


def predict(model, ds, collate, device):
    """Probabilities in the order of ds.ids (batches are formed by length for speed)."""
    model.eval()
    lengths = [len(ds.enc["input_ids"][j]) for j in ds.ids]
    out = np.zeros((len(ds), model.config.num_labels), dtype=np.float32)
    loader = DataLoader(ds, batch_sampler=sorted_batches(lengths, EVAL_BATCH), collate_fn=collate)
    order = [i for b in sorted_batches(lengths, EVAL_BATCH) for i in b]
    pos = 0
    with torch.no_grad():
        for b in loader:
            n = b["input_ids"].shape[0]
            b = {k: v.to(device) for k, v in b.items() if k != "labels"}
            with torch.autocast(device_type=device.type, dtype=amp_dtype(device), enabled=device.type == "cuda"):
                logits = model(**b).logits
            out[order[pos:pos + n]] = torch.softmax(logits.float(), -1).cpu().numpy()
            pos += n
    return out


# ----------------------------------------------------------------------------- training
def train_one(ckpt, ds_tr, collate, n_classes, lr, epochs, seed, device, eval_fn=None):
    """Trains for `epochs`; if eval_fn is given it is called after every epoch and the
    per-epoch outputs are returned (used for epoch selection)."""
    from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    torch.manual_seed(seed); np.random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=n_classes).to(device)
    lengths = [len(ds_tr.enc["input_ids"][j]) for j in ds_tr.ids]
    sampler = LengthGroupedBatchSampler(lengths, BATCH, seed)
    loader = DataLoader(ds_tr, batch_sampler=sampler, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WD)
    total = len(sampler) * epochs
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)
    dt = amp_dtype(device); scaler = torch.amp.GradScaler("cuda", enabled=(dt == torch.float16))
    history = []
    for ep in range(epochs):
        model.train(); t0 = time.time(); loss_sum = 0.0; n_tok = 0
        for b in loader:
            n_tok += int(b["attention_mask"].sum())
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast(device_type=device.type, dtype=dt, enabled=device.type == "cuda"):
                loss = model(**b).loss
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True); loss_sum += loss.item()
        secs = time.time() - t0
        rec = {"epoch": ep + 1, "train_loss": round(loss_sum / len(sampler), 4), "epoch_seconds": round(secs, 1),
               "docs_per_s": round(len(ds_tr) / secs, 1)}
        if eval_fn is not None:
            rec.update(eval_fn(model))
        history.append(rec); print("   ", rec, flush=True)
    return model, history


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="*", default=["dbmdz/bert-base-turkish-cased", "xlm-roberta-base"])
    ap.add_argument("--lrs", nargs="*", type=float, default=[2e-5, 5e-5])
    ap.add_argument("--field", default="text_clean")
    ap.add_argument("--cv", action="store_true")
    ap.add_argument("--max_epochs", type=int, default=MAX_EPOCHS)
    args = ap.parse_args()
    import transformers
    from transformers import AutoTokenizer, DataCollatorWithPadding
    transformers.logging.set_verbosity_error()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
    print("device:", device, "| precision:", amp_dtype(device), "| GPU:", torch.cuda.get_device_name(0) if device.type == "cuda" else "-", flush=True)
    docs, labels = load_docs(); var = load_variants(); idx = split_indices(docs)
    texts = var[args.field].fillna("").values
    y = np.array([labels.index(l) for l in docs["label"]]); y_str = docs["label"].values
    label_sets = docs["label_set"].values
    FT_DIR.mkdir(parents=True, exist_ok=True); PREDS.mkdir(parents=True, exist_ok=True)
    (RESULTS / "preds_val").mkdir(parents=True, exist_ok=True)

    for ckpt in args.models:
        short = "ft_" + ckpt.split("/")[-1]
        tok = AutoTokenizer.from_pretrained(ckpt)
        enc_all = tokenize(tok, texts)
        collate = DataCollatorWithPadding(tok)
        L = np.array([len(x) for x in enc_all["input_ids"]])
        print(f"[{short}] tokens per document: mean {L.mean():.0f}, median {np.median(L):.0f}, "
              f"truncated at {MAX_LEN}: {(L >= MAX_LEN).mean():.1%}", flush=True)
        mk = lambda ids: DS(enc_all, ids, y)
        ds_val = mk(idx["val"])
        prec = str(amp_dtype(device)).replace("torch.", "")

        # ---- 1. selection (seed 0): lr x epoch on the validation split
        sel_file = FT_DIR / f"{short}__selection.json"
        if sel_file.exists():
            sel = json.loads(sel_file.read_text())
        else:
            sel = {"runs": [], "precision": prec, "max_len": MAX_LEN, "batch": BATCH}
            for lr in args.lrs:
                def eval_fn(model):
                    p = predict(model, ds_val, collate, device)
                    m = metrics(y_str[idx["val"]], np.array(labels)[p.argmax(1)], labels)
                    return {"val_accuracy": round(m["accuracy"], 4), "val_macro_f1": round(m["macro_f1"], 4)}
                print(f"[{short}] selection lr={lr}", flush=True)
                _, hist = train_one(ckpt, mk(idx["train"]), collate, len(labels), lr, args.max_epochs, 0, device, eval_fn)
                sel["runs"].append({"lr": lr, "history": hist})
            best = max(((r["lr"], h["epoch"], h["val_macro_f1"]) for r in sel["runs"] for h in r["history"]), key=lambda t: t[2])
            sel["best"] = {"lr": best[0], "epochs": best[1], "val_macro_f1": best[2]}
            sel_file.write_text(json.dumps(sel, indent=2))
        lr, epochs = sel["best"]["lr"], sel["best"]["epochs"]
        print(f"[{short}] selected lr={lr} epochs={epochs} (val macro-F1={sel['best']['val_macro_f1']:.4f})", flush=True)

        # ---- 1b. validation predictions of the selected configuration (train only, seed 0) for deferral thresholds
        done_1b = FT_DIR / f"{short}__val_preds.done"
        if not done_1b.exists():
            model, _ = train_one(ckpt, mk(idx["train"]), collate, len(labels), lr, epochs, 0, device)
            P = predict(model, ds_val, collate, device)
            pd.to_pickle({"doc_id": docs["doc_id"].values[idx["val"]], "y_true": y_str[idx["val"]], "y_pred": np.array(labels)[P.argmax(1)],
                          "scores": P, "labels": labels, "params": {"lr": lr, "epochs": epochs}, "family": "FT_s0", "representation": short},
                         RESULTS / "preds_val" / f"{short}__FT_s0.pkl")
            done_1b.write_text("ok"); del model; torch.cuda.empty_cache()

        # ---- 2. final: train+val -> test, several seeds
        ds_test = mk(idx["test"])
        for seed in SEEDS_FINAL:
            tag = f"{short}__final_s{seed}"
            if (FT_DIR / f"{tag}.json").exists():
                print("done:", tag); continue
            t0 = time.time()
            model, hist = train_one(ckpt, mk(idx["trainval"]), collate, len(labels), lr, epochs, seed, device)
            fit_t = time.time() - t0
            t0 = time.time(); P = predict(model, ds_test, collate, device); pred_t = (time.time() - t0) / len(idx["test"])
            yp = np.array(labels)[P.argmax(1)]
            m = metrics(y_str[idx["test"]], yp, labels, scores=P, label_sets=label_sets[idx["test"]])
            ci = bootstrap_ci(y_str[idx["test"]], yp, labels)
            row = {"representation": short, "family": f"FT_s{seed}",
                   "params": json.dumps({"lr": lr, "epochs": epochs, "seed": seed, "max_len": MAX_LEN, "batch": BATCH, "precision": prec}),
                   **m, **ci, "fit_seconds_trainval": fit_t, "predict_ms_per_doc": pred_t * 1000, "n_features": 0,
                   "checkpoint": ckpt, "model_family": "fine-tuned encoder", "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu"}
            pd.to_pickle({"doc_id": docs["doc_id"].values[idx["test"]], "y_true": y_str[idx["test"]], "y_pred": yp, "scores": P,
                          "labels": labels, "params": row["params"], "family": row["family"], "representation": short}, PREDS / f"{short}__FT_s{seed}.pkl")
            append_results([row], file="results_ft.csv"); (FT_DIR / f"{tag}.json").write_text(json.dumps(row, indent=2))
            print(f"[{tag}] test acc={m['accuracy']:.4f} macroF1={m['macro_f1']:.4f}", flush=True)
            del model; torch.cuda.empty_cache()

        # ---- 3. optional grouped CV (seed-0 folds)
        if args.cv:
            accs, f1s = [], []
            for k, (tr, te) in enumerate(cv_folds(docs)[:5]):
                tag = f"{short}__cv_f{k}"
                f = FT_DIR / f"{tag}.json"
                if f.exists():
                    r = json.loads(f.read_text())
                else:
                    model, _ = train_one(ckpt, mk(tr), collate, len(labels), lr, epochs, 0, device)
                    P = predict(model, mk(te), collate, device)
                    m = metrics(y_str[te], np.array(labels)[P.argmax(1)], labels)
                    r = {"fold": k, "accuracy": m["accuracy"], "macro_f1": m["macro_f1"]}
                    f.write_text(json.dumps(r)); del model; torch.cuda.empty_cache()
                    print(f"[{tag}] acc={r['accuracy']:.4f} macroF1={r['macro_f1']:.4f}", flush=True)
                accs.append(r["accuracy"]); f1s.append(r["macro_f1"])
            cvr = {"cv_accuracy_mean": float(np.mean(accs)), "cv_accuracy_sd": float(np.std(accs, ddof=1)),
                   "cv_macro_f1_mean": float(np.mean(f1s)), "cv_macro_f1_sd": float(np.std(f1s, ddof=1))}
            (FT_DIR / f"{short}__cv_summary.json").write_text(json.dumps(cvr, indent=2))
            print(f"[{short}] CV:", cvr, flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/drive/MyDrive/ebys-routing/src/llm.py
"""
llm.py
------
Large-language-model routing experiments (API based; resumable; privacy-preserving).

Modes
  zeroshot : the model receives the list of the 13 units with short duty descriptions and
             the (de-identified) document, and returns the three most likely units, ranked.
  rag<k>   : additionally receives the k most similar past documents (TF-IDF cosine over the
             training pool: `train` for validation documents, `train+val` for test documents)
             together with the units they were dispatched to (retrieval-augmented few-shot).

Privacy: before any text leaves the machine, person names are replaced with [KİŞİ] by a
Turkish NER model (savasy/bert-base-turkish-ner-cased), and e-mail addresses, phone numbers
and 11-digit identity numbers are masked by regular expressions.

Every response is appended to results/llm/<model>__<mode>__<split>.jsonl as soon as it
arrives; documents already present in that file are skipped on re-runs.

    python src/llm.py --models claude-haiku-5-5 gemini-3.8-flash --modes zeroshot rag10 --splits test val
    python src/llm.py --smoke            # one document per model, prints raw responses
"""
import argparse
import json
import os
import re
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import PREDS, RESULTS, append_results, bootstrap_ci, load_docs, load_variants, metrics, split_indices

LLM_DIR = RESULTS / "llm"
MASKED = RESULTS / "llm" / "masked_texts.pkl"

# provider, input price, output price (USD per 1M tokens; edit to the current list prices)
MODELS = {
    "claude-haiku-5-5": ("anthropic", 0.10, 0.50),
    "claude-sonnet-5-5": ("anthropic", 2.00, 10.00),
    "gpt-6-luna": ("openai", 0.10, 0.50),
    "gpt-6.1-terra": ("openai", 1.00, 5.00),
    "gemini-3.8-flash": ("google", 0.75, 3.75),     # standard paid tier, Oct-Dec 2026 (verify current list price)
    "gemini-3.6-flash": ("google", None, None),
}

# Duty descriptions of the 13 target units (written from the units' functions and the
# document types observed in the training data; edit freely).
UNIT_DESCRIPTIONS = {
    "Rektörlük": "Üst yönetim; Yönetim Kurulu, Senato ve Disiplin Kurulu toplantı gündemleri, üst yönetime arz edilen yazılar, rektörlük oluru gerektiren işler.",
    "Bilgi İşlem Daire Başkanlığı": "BİDB'nin daire başkanlığı makamı; daire düzeyinde vekâlet, görevlendirme, anket, mezun bilgi sistemi ve merkez müdürlüğü yazışmaları, şube müdürlüklerine dağıtılacak genel bilgi işlem talepleri.",
    "Yazılım Şube Müdürlüğü": "Yazılım geliştirme ve bilgi sistemleri: EBYS, öğrenci/personel bilgi sistemi, web sayfası güncellemeleri, kullanıcı tanımlama; atama, göreve başlama, ayrılan personel, anabilim dalı/bölüm başkanlığı değişikliği gibi sistemde güncelleme gerektiren personel yazıları.",
    "Bilgi İşlem Şube Müdürlüğü": "Donanım ve teknik destek: bilgisayar, tablet, webcam, yazıcı talepleri, laboratuvar bilgisayarları, bakım-onarım, hurda ve demirbaş işlemleri, teknik malzeme irsaliyeleri.",
    "İnternet Hizmetleri Şube Müdürlüğü": "Ağ ve internet hizmetleri: internet erişimi, e-posta ve parola, IP adresi, kablolama, web alan adı, kamera ağ bağlantısı, teknopark/uzak bağlantı talepleri.",
    "Haberleşme Hizmetleri Şube Müdürlüğü": "Telefon ve haberleşme altyapısı: dahili/dış hat talepleri, santral, telefon arızaları, faks, şehirlerarası arama açma, kamera sistemi arızaları.",
    "Sağlık Kültür ve Spor Daire Başkanlığı": "SKS'nin daire başkanlığı makamı; daire düzeyinde vekâlet, görevlendirme, olur alınması, yolluk ve izin yazıları, şube müdürlüklerine dağıtılacak genel talepler.",
    "Kültür Hizmetleri Şube Müdürlüğü": "Kültürel etkinlikler: konferans salonu tahsisi, mezuniyet ve kuruluş yıldönümü törenleri, öğrenci toplulukları, geziler, tiyatro/konser, etkinlikler için araç tahsisi.",
    "Satın Alma Şube Müdürlüğü": "Satın alma ve mali işler: kısmi zamanlı öğrenci çalıştırma kontenjanları, başvuru, puantaj ve ödeme işlemleri; cübbe/şal gibi malzeme talepleri; ihale ve satın alma yazışmaları.",
    "Sosyal Tesisler Şube Müdürlüğü": "Sosyal tesisler ve kiralama: kantin, kafeterya, çay ocağı ve restoran işletmeleri, kira bedeli ve su birim fiyatı belirleme, otomat, dış sınavlara (AUZEF vb.) mekân tahsisi.",
    "Beslenme Hizmetleri Şube Müdürlüğü": "Beslenme hizmetleri: yemekhane ve yemek hizmeti, yemek bursu, kantin ve yemekhane denetleme komisyonu, yemek ihalesi ve maliyet hesapları, personel ek gösterge yazıları.",
    "Spor Tesisleri Şube Müdürlüğü": "Spor tesisleri ve faaliyetleri: üniversite içi turnuvalar, spor temsilcileri, hakem görevlendirme ve izinleri, havuz ve spor salonu işletmesi, havuz suyu analizleri.",
    "Sağlık Hizmetleri Şube Müdürlüğü": "Sağlık hizmetleri: sağlık merkezi, Covid-19 testleri, sağlık çalışma grupları, fiili hizmet zammı gibi sağlık personeli yazıları.",
}

SYSTEM = (
    "Sen bir üniversitenin Elektronik Belge Yönetim Sistemi'nde (EBYS) gelen evrakı, gereği için sevk edilmesi gereken "
    "birime yönlendiren bir asistansın. Evrakın konusu ve içeriği verilecektir. Aşağıdaki birim listesinden, evrakın "
    "gereği için sevk edilmesi en olası üç birimi, en olasıdan başlayarak sırala. Birim adlarını listede yazıldığı "
    "biçimde, harfi harfine kullan. Yalnızca JSON döndür: {\"birimler\": [\"birim1\", \"birim2\", \"birim3\"]}\n\n"
    "BİRİMLER VE GÖREV ALANLARI:\n" + "\n".join(f"- {u}: {d}" for u, d in UNIT_DESCRIPTIONS.items())
)

def json_schema():
    """Structured-output schema: an array of unit names restricted to the 13 labels."""
    return {"type": "object",
            "properties": {"birimler": {"type": "array", "items": {"type": "string", "enum": list(UNIT_DESCRIPTIONS)}}},
            "required": ["birimler"], "additionalProperties": False}


# ----------------------------------------------------------------------------- de-identification
def _mask_patterns(t):
    t = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", "[E-POSTA]", t)
    t = re.sub(r"(?<!\d)\d{11}(?!\d)", "[TCKN]", t)
    t = re.sub(r"(?<!\d)(?:\+90|0)?\s?\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2}(?!\d)", "[TELEFON]", t)
    return t


def mask_texts(texts, device=None, words_per_chunk=120):
    """Replace person names ([KİŞİ]), e-mails, phone and 11-digit numbers before API calls.
    Texts are cut into windows of `words_per_chunk` words (well below the 512-token limit of
    the NER model); masked windows are re-joined with single spaces."""
    import torch
    from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline
    device = device if device is not None else (0 if torch.cuda.is_available() else -1)
    ckpt = "savasy/bert-base-turkish-ner-cased"
    tok = AutoTokenizer.from_pretrained(ckpt)
    tok.model_max_length = 512
    ner = pipeline("ner", model=AutoModelForTokenClassification.from_pretrained(ckpt), tokenizer=tok,
                   aggregation_strategy="simple", device=device)
    # flatten into chunks
    chunks, owner = [], []
    for n, t in enumerate(texts):
        words = str(t).split()
        if not words:
            chunks.append(""); owner.append(n); continue
        for i in range(0, len(words), words_per_chunk):
            chunks.append(" ".join(words[i:i + words_per_chunk])); owner.append(n)
    masked_chunks = []
    bs = 64
    for i in range(0, len(chunks), bs):
        batch = chunks[i:i + bs]
        nonempty = [c for c in batch if c.strip()]
        ents = iter(ner(nonempty, batch_size=32) if nonempty else [])
        for c in batch:
            if not c.strip():
                masked_chunks.append(c); continue
            es = next(ents)
            spans = sorted([(e["start"], e["end"]) for e in es if e.get("entity_group") == "PER"], reverse=True)
            for s_, e_ in spans:
                c = c[:s_] + "[KİŞİ]" + c[e_:]
            masked_chunks.append(c)
        if (i // bs) % 20 == 0:
            print(f"   masked {min(i + bs, len(chunks))}/{len(chunks)} chunks", flush=True)
    out = [[] for _ in texts]
    for n, c in zip(owner, masked_chunks):
        out[n].append(c)
    return [_mask_patterns(" ".join(parts)) for parts in out]


def get_masked(docs, var):
    if MASKED.exists():
        m = pd.read_pickle(MASKED)
        if len(m) == len(docs):
            return m
    LLM_DIR.mkdir(parents=True, exist_ok=True)
    print("de-identifying texts with Turkish NER ...")
    m = pd.Series(mask_texts(var["text_clean"].fillna("").tolist()), index=docs.index)
    m.to_pickle(MASKED)
    return m


# ----------------------------------------------------------------------------- retrieval
def build_retriever(texts_pool):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")
    M = vec.fit_transform(texts_pool)
    return vec, M


def neighbours(vec, M, text, k):
    q = vec.transform([text])
    sims = (M @ q.T).toarray().ravel()
    return np.argsort(-sims)[:k], sims


# ----------------------------------------------------------------------------- providers
_clients = {}
_lock = threading.Lock()
_THINKING = {}      # Claude model -> accepted "thinking off" parameter
_CFG = {}           # model -> effective reasoning/thinking setting of the last successful call (stored per record)


def _client(provider):
    with _lock:
        if provider in _clients:
            return _clients[provider]
        if provider == "anthropic":
            import anthropic
            c = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY") or os.environ["CLAUDE_API_KEY"])
        elif provider == "openai":
            import openai
            c = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        elif provider == "google":
            from google import genai
            c = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY") or os.environ["GEMINI_API_KEY"])
        _clients[provider] = c
        return c


def model_info(model):
    """(provider, price_in, price_out); unknown models are assigned a provider by name prefix."""
    if model in MODELS:
        return MODELS[model]
    low = model.lower()
    prov = "anthropic" if low.startswith("claude") else "openai" if (low.startswith("gpt") or low.startswith("o")) else "google" if low.startswith("gemini") else None
    if prov is None:
        raise ValueError(f"unknown provider for model {model}; add it to llm.MODELS")
    return (prov, None, None)


def call_model(model, user_text, max_tokens=300):
    """Returns (raw_text, usage_in, usage_out).  Reasoning/thinking is kept minimal."""
    provider = model_info(model)[0]
    c = _client(provider)
    if provider == "anthropic":
        kw = dict(model=model, max_tokens=max_tokens, system=SYSTEM, messages=[{"role": "user", "content": user_text}])
        # "Thinking off" is spelled differently across Claude models: {"type": "disabled"} (e.g. Haiku 5.5) or
        # {"type": "between_tools"} (e.g. Sonnet 5.5, which rejects "disabled": thinking only between tool
        # calls, i.e. none for this tool-free request).  The accepted form is cached per model.  The
        # parameter is never omitted: the default of some models is adaptive thinking, which can use up
        # the answer budget (observed for Sonnet 5.5 in the first pass).
        tries = [_THINKING[model]] if model in _THINKING else [{"type": "disabled"}, {"type": "between_tools"}]
        last = None
        for th in tries:
            try:
                r = c.messages.create(**kw, thinking=th)
                _THINKING[model] = th; _CFG[model] = {"thinking": th["type"]}
                return "".join(getattr(b, "text", "") for b in r.content if getattr(b, "type", "") == "text"), r.usage.input_tokens, r.usage.output_tokens
            except Exception as e:
                if "thinking" not in str(e).lower():
                    raise
                last = e
        raise last
    if provider == "openai":
        try:
            r = c.responses.create(model=model, instructions=SYSTEM, input=user_text, max_output_tokens=max_tokens,
                                   reasoning={"effort": "none"},
                                   text={"format": {"type": "json_schema", "name": "routing", "schema": json_schema(), "strict": True}})
            _CFG[model] = {"reasoning_effort": "none"}
            return (r.output_text or ""), r.usage.input_tokens, r.usage.output_tokens
        except Exception as e:
            if "reasoning" in str(e).lower() or "effort" in str(e).lower():
                r = c.responses.create(model=model, instructions=SYSTEM, input=user_text, max_output_tokens=max(max_tokens, 2048),
                                       text={"format": {"type": "json_schema", "name": "routing", "schema": json_schema(), "strict": True}})
                _CFG[model] = {"reasoning_effort": "default"}
                return (r.output_text or ""), r.usage.input_tokens, r.usage.output_tokens
            raise
    if provider == "google":
        from google.genai import types
        base = dict(system_instruction=SYSTEM, temperature=0, max_output_tokens=max_tokens, response_mime_type="application/json")
        try:
            cfg = types.GenerateContentConfig(**base, thinking_config=types.ThinkingConfig(thinking_budget=0))
            r = c.models.generate_content(model=model, contents=user_text, config=cfg)
            _CFG[model] = {"thinking_budget": 0, "temperature": 0}
        except Exception as e:
            if "thinking" not in str(e).lower():
                raise
            r = c.models.generate_content(model=model, contents=user_text, config=types.GenerateContentConfig(**base))
            _CFG[model] = {"thinking_budget": "default", "temperature": 0}
        um = r.usage_metadata
        txt = None
        try:
            txt = r.text
        except Exception:
            txt = None
        return (txt or ""), (um.prompt_token_count or 0) if um else 0, (um.candidates_token_count or 0) if um else 0
    raise ValueError(provider)


_ASCII = str.maketrans("çğıöşüÇĞİÖŞÜâîû", "cgiosuCGIOSUaiu")


def _norm(s):
    return re.sub(r"[^a-z0-9 ]+", " ", s.translate(_ASCII).lower()).strip()


MIN_NAME_SIMILARITY = 0.85      # spelling variants ("Sağlık, Kültür ve ...") are kept; other units are not


def parse_units(raw, labels, min_sim=MIN_NAME_SIMILARITY):
    """Map the model answer to at most three unit names, in the model's order.

    1. The first complete JSON object in the answer is used (text or a repeated object after it is ignored).
    2. A truncated object keeps the complete quoted names that follow the "birimler" key.
    3. Anything else (refusal, free text, empty answer) is no answer: [].
    A name that is not literally a unit is mapped to the most similar unit (accent-insensitive
    difflib ratio) only if the similarity is >= min_sim; names of units outside the list
    (e.g. "Personel Daire Başkanlığı") are discarded."""
    import difflib
    raw = raw or ""
    cands = []
    s = raw.find("{")
    if s >= 0:
        try:
            obj, _ = json.JSONDecoder().raw_decode(raw[s:])
            cands = obj.get("birimler", []) if isinstance(obj, dict) else []
        except Exception:
            k = raw.find('"birimler"', s)
            if k >= 0:
                cands = re.findall(r'"([^"]+)"', raw[k + len('"birimler"'):])
    if isinstance(cands, str):
        cands = [cands]
    out = []
    norm_labels = {l: _norm(l) for l in labels}
    for cnd in cands if isinstance(cands, list) else []:
        if not isinstance(cnd, str):
            continue
        if cnd in labels:
            u = cnd
        else:
            sims = {l: difflib.SequenceMatcher(None, _norm(cnd), norm_labels[l]).ratio() for l in labels}
            u = max(sims, key=sims.get)
            if sims[u] < min_sim:
                continue
        if u not in out:
            out.append(u)
    return out[:3]


def out_of_list(raw, labels, min_sim=MIN_NAME_SIMILARITY):
    """True if the answer names at least one unit that is not in the list (diagnostic)."""
    s = (raw or "").find("{")
    try:
        obj, _ = json.JSONDecoder().raw_decode(raw[s:]) if s >= 0 else ({}, 0)
        c = obj.get("birimler", []) if isinstance(obj, dict) else []
    except Exception:
        return False
    return any(isinstance(x, str) and x not in labels and not parse_units(json.dumps({"birimler": [x]}), labels, min_sim)
               for x in (c if isinstance(c, list) else []))


def read_records(f):
    """All records of a jsonl file in file order (a line cut by an interruption is skipped)."""
    recs = []
    if Path(f).exists():
        for l in Path(f).read_text(encoding="utf-8").splitlines():
            try:
                recs.append(json.loads(l))
            except Exception:
                pass
    return recs


def final_records(f):
    """Last record per document: a re-requested answer (pass 2) supersedes the first one."""
    out = {}
    for r in read_records(f):
        out[r["doc_id"]] = r
    return out


def final_answers(model, mode, split, labels):
    """doc_id -> parsed units of the final record (None if the file does not exist)."""
    f = LLM_DIR / f"{model}__{mode}__{split}.jsonl"
    if not f.exists():
        return None
    return {d: parse_units(r.get("raw", ""), labels) for d, r in final_records(f).items()}


def hidden_output_tokens(rec):
    """Output tokens not accounted for by the visible answer (Claude tokenizer: 5.4 + 0.623 tokens per
    character, fitted on Claude Haiku 5.5 answers without thinking; residual SD 1.6 tokens).  Values
    well above zero mean that the model produced hidden thinking before the answer."""
    return rec["out_tokens"] - (5.4 + 0.623 * len(rec.get("raw") or ""))


# ----------------------------------------------------------------------------- main loop
def build_user_text(subject_text, examples):
    s = ""
    if examples:
        s += "GEÇMİŞ EVRAKLAR VE SEVK EDİLDİKLERİ BİRİMLER (benzerlik sırasına göre):\n"
        for j, (t, u) in enumerate(examples, 1):
            s += f"{j}. Birim: {u}\n   Evrak: {t[:700]}\n"
        s += "\n"
    s += f"YÖNLENDİRİLECEK EVRAK:\n{subject_text[:3000]}\n\nJSON:"
    return s


# which splits each mode is run on: zero-shot only needs the test split; the retrieval-augmented
# mode is also run on validation because the cascade threshold is chosen there
PLAN = {"zeroshot": ["test"], "rag10": ["test", "val"]}


def run(models, modes, splits=None, workers=6, max_docs=None, smoke=False, max_consecutive_failures=15):
    """Returns a list of (model, mode, split) combinations that failed or were aborted."""
    docs, labels = load_docs(); var = load_variants(); idx = split_indices(docs)
    masked = get_masked(docs, var)
    LLM_DIR.mkdir(parents=True, exist_ok=True)
    all_splits = splits or sorted({s for m in modes for s in PLAN.get(m, ["test", "val"])})
    failed = []
    for split in all_splits:
        target = idx[split][:max_docs] if max_docs else idx[split]
        for mode in modes:
            if splits is None and split not in PLAN.get(mode, ["test", "val"]):
                continue
            k = int(mode[3:]) if mode.startswith("rag") else 0
            # neighbours are computed once per document (shared by all models) and cached
            nbs = _neighbours(mode, split, docs, var, idx) if k else None
            for model in models:
                f = LLM_DIR / f"{model}__{mode}__{split}.jsonl"
                done = set()
                if f.exists():
                    for l in f.read_text(encoding="utf-8").splitlines():
                        try:
                            done.add(json.loads(l)["doc_id"])
                        except Exception:
                            pass          # a line cut by an interruption is ignored and redone
                todo = [i for i in target if int(docs["doc_id"].values[i]) not in done]
                if smoke:
                    todo = todo[:1]
                print(f"[{model} | {mode} | {split}] {len(done)} done, {len(todo)} to do", flush=True)
                if not todo:
                    continue
                lock = threading.Lock()
                state = {"consecutive": 0, "abort": False, "last_error": ""}

                def work(i):
                    if state["abort"]:
                        return False
                    did = int(docs["doc_id"].values[i])
                    ex = [(masked.iloc[j], docs["label"].values[j]) for j in nbs[did]] if k else []
                    user = build_user_text(masked.iloc[i], ex)
                    for attempt in range(6):
                        if state["abort"]:
                            return False
                        try:
                            t0 = time.time(); raw, tin, tout = call_model(model, user); lat = time.time() - t0
                            rec = {"doc_id": did, "raw": raw, "units": parse_units(raw, labels), "in_tokens": tin, "out_tokens": tout,
                                   "latency_s": lat, "ts": time.time(), "cfg": _CFG.get(model)}
                            with lock:
                                with open(f, "a", encoding="utf-8") as fh:
                                    fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
                                state["consecutive"] = 0
                            if smoke:
                                print(f"--- {model} RAW: {raw[:300]}\n    parsed: {rec['units']}  true: {docs['label'].values[i]}", flush=True)
                            return True
                        except Exception as e:
                            state["last_error"] = str(e)[:300]
                            if smoke and attempt >= 1:
                                break
                            wait = 2 ** attempt
                            print(f"  retry {attempt+1} for doc {did} ({model}): {str(e)[:160]} -> sleeping {wait}s", flush=True)
                            time.sleep(wait)
                    with lock:
                        state["consecutive"] += 1
                        if state["consecutive"] >= max_consecutive_failures:
                            state["abort"] = True
                    return False

                with ThreadPoolExecutor(max_workers=workers) as ex_:
                    futs = [ex_.submit(work, i) for i in todo]
                    n_ok = 0
                    for n, fu in enumerate(as_completed(futs), 1):
                        n_ok += fu.result()
                        if n % 200 == 0:
                            print(f"   {model} | {mode} | {split}: {n}/{len(todo)} processed", flush=True)
                print(f"   finished {n_ok}/{len(todo)}", flush=True)
                if state["abort"] or n_ok < len(todo):
                    failed.append((model, mode, split))
                    msg = "ABORTED after repeated failures" if state["abort"] else f"{len(todo) - n_ok} documents failed"
                    print(f"!! [{model} | {mode} | {split}] {msg}. Last error: {state['last_error']}", flush=True)
    return failed


def _neighbours(mode, split, docs, var, idx):
    """Retrieved examples (positions in `docs`) for every document of the split; cached on disk."""
    k = int(mode[3:])
    nb_file = LLM_DIR / f"neighbours__{mode}__{split}.pkl"
    nbs = pd.read_pickle(nb_file) if nb_file.exists() else None
    if nbs is None or any(int(d) not in nbs for d in docs["doc_id"].values[idx[split]]):
        pool = idx["train"] if split == "val" else idx["trainval"]
        vec, M = build_retriever(var["text_raw"].values[pool])
        nbs = {int(docs["doc_id"].values[i]): [int(pool[j]) for j in neighbours(vec, M, var["text_raw"].values[i], k)[0]]
               for i in idx[split]}
        pd.to_pickle(nbs, nb_file)
    return nbs


def requery(models, modes, workers=6, max_attempts=3, hidden_threshold=20, max_consecutive_failures=15):
    """Second pass over finished runs.  A document is requested again when its final answer
      (i)  contains no valid unit (truncated JSON, refusal, empty answer, only units outside the list), or
      (ii) for Claude models, was preceded by hidden thinking (hidden_output_tokens > hidden_threshold);
           the first Claude Sonnet 5.5 pass ran with the model's default adaptive thinking.
    At most `max_attempts` requests per document, stopping at the first answer with a valid unit.  Every
    response is appended to the same file with "pass": 2, so the last record of a document is its final
    answer and every call is counted in the cost.  Documents that already have a pass-2 record are skipped."""
    docs, labels = load_docs(); var = load_variants(); idx = split_indices(docs)
    masked = get_masked(docs, var)
    pos = {int(d): i for i, d in enumerate(docs["doc_id"].values)}
    failed = []
    for mode in modes:
        k = int(mode[3:]) if mode.startswith("rag") else 0
        for split in PLAN.get(mode, ["test", "val"]):
            nbs = _neighbours(mode, split, docs, var, idx) if k else None
            for model in models:
                f = LLM_DIR / f"{model}__{mode}__{split}.jsonl"
                if not f.exists():
                    continue
                provider = model_info(model)[0]
                fin = final_records(f)
                missing = set(int(d) for d in docs["doc_id"].values[idx[split]]) - set(fin)
                if missing:
                    print(f"[{model} | {mode} | {split}] first pass incomplete ({len(missing)} missing): run it first", flush=True)
                    continue
                redo = []
                for d, r in fin.items():
                    if r.get("pass") == 2:
                        continue
                    if provider == "anthropic" and hidden_output_tokens(r) > hidden_threshold:
                        redo.append((d, "hidden thinking"))
                    elif not parse_units(r.get("raw", ""), labels):
                        redo.append((d, "no valid unit"))
                n_th = sum(1 for _, why in redo if why == "hidden thinking")
                print(f"[requery {model} | {mode} | {split}] {len(redo)} documents ({n_th} hidden thinking, {len(redo) - n_th} no valid unit)", flush=True)
                if not redo:
                    continue
                lock = threading.Lock()
                state = {"consecutive": 0, "abort": False, "last_error": "", "valid": 0}

                def work(item):
                    d, why = item
                    i = pos[d]
                    ex = [(masked.iloc[j], docs["label"].values[j]) for j in nbs[d]] if k else []
                    user = build_user_text(masked.iloc[i], ex)
                    for attempt in range(1, max_attempts + 1):
                        raw = None
                        for net_try in range(6):
                            if state["abort"]:
                                return False
                            try:
                                t0 = time.time(); raw, tin, tout = call_model(model, user); lat = time.time() - t0
                                break
                            except Exception as e:
                                state["last_error"] = str(e)[:300]
                                time.sleep(2 ** net_try)
                        if raw is None:
                            with lock:
                                state["consecutive"] += 1
                                if state["consecutive"] >= max_consecutive_failures:
                                    state["abort"] = True
                            return False
                        units = parse_units(raw, labels)
                        rec = {"doc_id": d, "raw": raw, "units": units, "in_tokens": tin, "out_tokens": tout, "latency_s": lat,
                               "ts": time.time(), "cfg": _CFG.get(model), "pass": 2, "attempt": attempt, "reason": why}
                        with lock:
                            with open(f, "a", encoding="utf-8") as fh:
                                fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
                            state["consecutive"] = 0
                        if units:
                            with lock:
                                state["valid"] += 1
                            return True
                    return True          # all attempts answered without a valid unit: kept as no answer

                # one request first: a model that cannot be called without thinking (or a key problem) stops here
                n_ok = int(work(redo[0]))
                if not n_ok:
                    failed.append((model, mode, split))
                    print(f"!! [requery {model}] first request failed: {state['last_error']}", flush=True)
                    continue
                with ThreadPoolExecutor(max_workers=workers) as ex_:
                    futs = [ex_.submit(work, it) for it in redo[1:]]
                    for n, fu in enumerate(as_completed(futs), 2):
                        n_ok += fu.result()
                        if n % 100 == 0:
                            print(f"   requery {model} | {mode} | {split}: {n}/{len(redo)}", flush=True)
                print(f"   finished {n_ok}/{len(redo)}; with a valid unit: {state['valid']}", flush=True)
                if state["abort"] or n_ok < len(redo):
                    failed.append((model, mode, split))
                    print(f"!! [requery {model} | {mode} | {split}] {len(redo) - n_ok} documents not done (re-run to resume). "
                          f"Last error: {state['last_error']}", flush=True)
                if provider == "anthropic":
                    p2 = [r for r in read_records(f) if r.get("pass") == 2]
                    print(f"   check: pass-2 answers with hidden thinking: {sum(hidden_output_tokens(r) > hidden_threshold for r in p2)}/{len(p2)}; "
                          f"setting: {sorted({json.dumps(r.get('cfg')) for r in p2})}", flush=True)
    return failed


DECODING = {   # reported in the params column
    "anthropic": {"thinking": "off (disabled; between_tools for models that reject disabled)", "temperature": "provider default", "max_tokens": 300},
    "openai": {"reasoning_effort": "none", "structured_output": "strict JSON schema, enum of units", "temperature": "provider default", "max_tokens": 300},
    "google": {"thinking_budget": 0, "temperature": 0, "response_mime_type": "application/json", "max_tokens": 300},
}


def summarise(models, modes, split="test"):
    """Metrics + prediction files in the common format for every finished (model, mode).  Answers are
    re-parsed from the raw text of the final record of each document (see parse_units)."""
    docs, labels = load_docs(); idx = split_indices(docs)
    fallback = docs["label"].values[idx["trainval"]]
    fallback = pd.Series(fallback).value_counts().index[0]      # majority unit of the training data
    test_ids = docs["doc_id"].values[idx[split]]
    y_true = docs["label"].values[idx[split]]; label_sets = docs["label_set"].values[idx[split]]
    rows = []
    for model in models:
        for mode in modes:
            f = LLM_DIR / f"{model}__{mode}__{split}.jsonl"
            if not f.exists():
                continue
            recs = final_records(f)
            if len(set(test_ids) - set(recs)) > 0:
                print(f"[{model} | {mode}] incomplete: {len(set(test_ids) - set(recs))} missing"); continue
            prov, pin, pout = model_info(model)
            scores = np.zeros((len(test_ids), len(labels)))
            y_pred = []
            parsed = {did: parse_units(recs[did].get("raw", ""), labels) for did in test_ids}
            n_empty = sum(1 for did in test_ids if not parsed[did])
            for n, did in enumerate(test_ids):
                units = parsed[did] or [fallback]
                for rank, u in enumerate(units):
                    scores[n, labels.index(u)] = 3 - rank
                y_pred.append(units[0])
            y_pred = np.array(y_pred)
            m = metrics(y_true, y_pred, labels, scores=scores, label_sets=label_sets)
            ci = bootstrap_ci(y_true, y_pred, labels)
            fin = [recs[did] for did in test_ids]
            alls = [r for r in read_records(f) if r["doc_id"] in recs]
            tin = sum(r["in_tokens"] for r in fin); tout = sum(r["out_tokens"] for r in fin)
            tin_all = sum(r["in_tokens"] for r in alls); tout_all = sum(r["out_tokens"] for r in alls)
            cost = (tin * pin + tout * pout) / 1e6 if pin is not None else None
            cost_all = (tin_all * pin + tout_all * pout) / 1e6 if pin is not None else None
            lat = float(np.median([r["latency_s"] for r in fin]))
            name = f"llm_{model}"
            row = {"representation": name, "family": mode, "params": json.dumps({"mode": mode, **DECODING[prov]}), **m, **ci,
                   "fit_seconds_trainval": 0.0, "predict_ms_per_doc": lat * 1000, "n_features": 0, "model_family": "LLM (" + prov + ")",
                   "in_tokens": tin, "out_tokens": tout, "usd_cost": cost, "usd_per_1000_docs": (cost / len(test_ids) * 1000) if cost else None,
                   "usd_cost_all_calls": cost_all, "n_calls": len(alls),
                   "n_empty_answers": n_empty, "n_requeried": sum(1 for r in fin if r.get("pass") == 2),
                   "n_out_of_list": sum(1 for r in fin if out_of_list(r.get("raw", ""), labels)),
                   "n_hidden_thinking": sum(1 for r in fin if hidden_output_tokens(r) > 20) if prov == "anthropic" else 0}
            pd.to_pickle({"doc_id": test_ids, "y_true": y_true, "y_pred": y_pred, "scores": scores, "labels": labels,
                          "params": row["params"], "family": mode, "representation": name}, PREDS / f"{name}__{mode}.pkl")
            rows.append(row)
            print(f"[{model} | {mode}] acc={m['accuracy']:.4f} macroF1={m['macro_f1']:.4f} top3={m['top3_accuracy']:.4f} "
                  f"empty={n_empty} requeried={row['n_requeried']} hidden_thinking={row['n_hidden_thinking']} "
                  f"cost=${cost if cost is None else round(cost, 3)} (all calls ${cost_all if cost_all is None else round(cost_all, 3)})")
    append_results(rows, file="results_llm.csv")
    return rows


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="*", default=["claude-haiku-5-5", "claude-sonnet-5-5", "gpt-6-luna", "gemini-3.8-flash"])
    ap.add_argument("--modes", nargs="*", default=["zeroshot", "rag10"])
    ap.add_argument("--splits", nargs="*", default=None, help="default: zeroshot -> test; rag10 -> test and val")
    ap.add_argument("--workers", type=int, default=6)
    ap.add_argument("--max_docs", type=int, default=None)
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--summarise", action="store_true")
    ap.add_argument("--requery", action="store_true", help="second pass: re-request answers without a valid unit and "
                    "Claude answers produced with hidden thinking (see requery)")
    ap.add_argument("--mask_only", action="store_true", help="only build the de-identified texts (GPU recommended)")
    a = ap.parse_args()
    if a.mask_only:
        d_, _ = load_docs(); get_masked(d_, load_variants()); print("masked texts ready:", MASKED)
    elif a.summarise:
        summarise(a.models, a.modes)
    elif a.requery:
        bad = requery(a.models, a.modes, a.workers)
        if bad:
            print("\nREQUERY not finished for:", bad, "- run the same command again to resume")
            sys.exit(1)
    else:
        failed = run(a.models, a.modes, a.splits, a.workers, a.max_docs, a.smoke)
        if a.smoke:
            bad = sorted({m for m, _, _ in failed})
            status = {m: ("failed" if m in bad else "ok") for m in a.models}
            (LLM_DIR / "smoke_status.json").write_text(json.dumps(status, indent=2))
            print("\nSMOKE TEST:", status)
            if bad:
                print("Failed models are skipped; check the model name (LLM_MODELS / src/llm.py MODELS), the API key and the quota.")
            if len(bad) == len(a.models):
                sys.exit(1)


In [ ]:
%%writefile /content/drive/MyDrive/ebys-routing/src/analysis.py
"""
analysis.py
-----------
Turns the stored predictions into the tables and figures of the paper.

    python src/analysis.py            # everything that is available so far

Outputs (results/ and figures/):
  tables.xlsx                 all tables (one sheet each) + CSV copies in results/tables/
  fig_dataset.{png,pdf}       class distribution and monthly volume with the split boundaries
  fig_main.{png,pdf}          test accuracy and macro-F1 (95% bootstrap CI) per selected system
  fig_confusion.{png,pdf}     row-normalised confusion matrix of the final system
  fig_perclass.{png,pdf}      per-unit F1 vs. training support for the main systems
  fig_deferral.{png,pdf}      coverage-accuracy curves (human deferral) and the LLM cascade
  fig_ablation.{png,pdf}      preprocessing / input-field / translation / context ablations
  fig_pipeline.{png,pdf}      framework diagram (Graphviz)
"""
import json
import re
import subprocess
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import colors as mcolors

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import FT_RUNS, PARENT, PREDS, RESULTS, ROOT, SHORT, load_all_results, load_docs, mcnemar_test, metrics, per_class_table, split_indices

FIG = ROOT / "figures"
TAB = RESULTS / "tables"
PREDS_VAL = RESULTS / "preds_val"

# --- palette (validated with the dataviz validator; light surface)
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "violet": "#6250d6", "yellow": "#eda100",
     "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#898781", "grid": "#e1e0d9", "axis": "#c3c2b7", "surface": "#fcfcfb"}
FAMILY_COLOR = {"Lexical (TF-IDF)": C["blue"], "Frozen encoder": C["aqua"], "Fine-tuned encoder": C["orange"], "LLM": C["violet"]}
SEQ = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": C["axis"], "axes.labelcolor": C["ink2"],
                     "xtick.color": C["ink2"], "ytick.color": C["ink2"], "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": False, "figure.facecolor": "white", "axes.facecolor": "white", "savefig.dpi": 300,
                     "legend.frameon": False, "axes.titlesize": 9.5, "axes.titleweight": "bold", "axes.titlelocation": "left"})

PRETTY = {
    "tfidf_word_raw": "TF-IDF word (raw)", "tfidf_word_stem": "TF-IDF word (stem)", "tfidf_word_lemma": "TF-IDF word (lemma)",
    "tfidf_char_raw": "TF-IDF char 2-5", "tfidf_wordchar_raw": "TF-IDF word+char", "tfidf_content_tr": "TF-IDF content (TR)",
    "tfidf_content_en": "TF-IDF content (EN, translated)", "tfidf_subject_raw": "TF-IDF subject only",
    "emb_minilm_multi": "paraphrase-multilingual-MiniLM-L12", "emb_mpnet_multi": "paraphrase-multilingual-mpnet-base",
    "emb_distiluse_multi": "distiluse-base-multilingual (mUSE)", "emb_e5_base": "multilingual-e5-base", "emb_e5_large": "multilingual-e5-large",
    "emb_labse": "LaBSE", "emb_sbert_tr": "BERTurk-SBERT (NLI/STS-tr)", "emb_berturk_mean": "BERTurk mean-pooled (frozen)",
    "emb_bge_m3": "BGE-M3", "emb_mpnet_en": "all-mpnet-base-v2 (EN, translated)", "emb_qwen3_0_6b": "Qwen3-Embedding-0.6B",
    "emb_qwen3_4b": "Qwen3-Embedding-4B", "ft_bert-base-turkish-cased": "BERTurk fine-tuned", "ft_xlm-roberta-base": "XLM-R base fine-tuned",
}


SUFFIX = {"content_clean": "content only, TR", "content_en_clean": "content only, EN translated"}
LLM_NAMES = {"llm_claude-haiku-5-5": "Claude Haiku 5.5", "llm_claude-sonnet-5-5": "Claude Sonnet 5.5", "llm_gpt-6-luna": "GPT-6 Luna",
             "llm_gemini-3.8-flash": "Gemini 3.8 Flash"}
FAMILY_LABEL = {"rag10": "RAG-10", "zeroshot": "zero-shot", "FT_s0": "seed 0", "FT_s1": "seed 1", "FT_s2": "seed 2"}


def fam_label(f):
    return FAMILY_LABEL.get(str(f), str(f))


def llm_label(key):
    """'llm_gemini-3.8-flash__rag10' -> 'Gemini 3.8 Flash / RAG-10'"""
    rep, _, fam = key.partition("__")
    return LLM_NAMES.get(rep, rep) + (" / " + fam_label(fam) if fam else "")


def pretty(rep):
    extra = ""
    if "+sender" in rep:
        rep = rep.replace("+sender", ""); extra = " + sender unit"
    base, _, suffix = rep.partition("__")
    name = PRETTY.get(base, LLM_NAMES.get(base, base.replace("llm_", "LLM: ")))
    if suffix:
        name += " [" + SUFFIX.get(suffix, suffix) + "]"
    return name + extra


def family_of(rep):
    if rep.startswith("tfidf"):
        return "Lexical (TF-IDF)"
    if rep.startswith("emb"):
        return "Frozen encoder"
    if rep.startswith("ft_"):
        return "Fine-tuned encoder"
    return "LLM"


ABLATION_REPS = {"tfidf_subject_raw", "tfidf_content_tr", "tfidf_content_en"}


def is_ablation(rep):
    """Input/context ablations are reported separately from the main comparison
    (main = subject + body as input, no sender information)."""
    return ("+sender" in rep) or ("__content" in rep) or (rep in ABLATION_REPS)


def load_results():
    df = load_all_results()
    # pilot rows: linear SVM on dense embeddings (MiniLM, distiluse) before it was restricted to sparse TF-IDF
    df = df[~(df["representation"].str.startswith("emb_") & (df["family"] == "LinearSVM"))].reset_index(drop=True)
    df["system"] = df["representation"].map(pretty) + " / " + df["family"].map(fam_label)
    df["group"] = df["representation"].map(family_of)
    return df


def val_selected_family(rep):
    """Family chosen on the validation split (best val macro-F1 over the whole grid)."""
    best, fam = -1, None
    for f in (RESULTS / "val_grid").glob(f"{rep}__*.csv"):
        g = pd.read_csv(f)
        if g["macro_f1"].max() > best:
            best, fam = g["macro_f1"].max(), f.stem.split("__")[-1]
    return fam, best


def llm_val_macro_f1(model, mode, docs, labels):
    """Validation macro-F1 of an LLM configuration (None if the validation run is incomplete)."""
    from sklearn.metrics import f1_score
    from llm import final_answers
    recs = final_answers(model, mode, "val", labels)
    if recs is None:
        return None
    v = docs[docs["split"] == "val"]
    if set(v["doc_id"]) - set(recs):
        return None
    fallback = docs.loc[docs["split"] != "test", "label"].value_counts().index[0]
    pred = [(recs[d] or [fallback])[0] for d in v["doc_id"]]
    return float(f1_score(v["label"], pred, labels=labels, average="macro", zero_division=0))


def selected_systems(df, docs=None, labels=None):
    """One row per representation: the classifier family selected on validation.
    Fine-tuned encoders: seed-averaged row (predictions of seed 0 are used for tests), validation
    macro-F1 of the selected epoch, grouped CV from ft_runs/.  LLMs: one row per mode."""
    rows = []
    for rep, g in df.groupby("representation", sort=False):
        if rep.startswith("ft_"):
            r = g.iloc[0].copy()
            for c in ["accuracy", "macro_f1", "weighted_f1", "balanced_accuracy", "top3_accuracy", "lenient_accuracy", "department_accuracy"]:
                r[c] = g[c].mean(); r[c + "_sd"] = g[c].std(ddof=1) if len(g) > 1 else np.nan
            r["family"] = "mean of %d seeds" % len(g); r["pred_key"] = f"{rep}__FT_s0"
            sel_f = FT_RUNS / f"{rep}__selection.json"
            if sel_f.exists():
                r["val_macro_f1"] = json.loads(sel_f.read_text())["best"]["val_macro_f1"]
            cv_f = FT_RUNS / f"{rep}__cv_summary.json"
            if cv_f.exists():
                for k, v in json.loads(cv_f.read_text()).items():
                    r[k] = v
            rows.append(r)
        elif rep.startswith("llm_"):
            for _, r in g.iterrows():
                r = r.copy(); r["pred_key"] = f"{rep}__{r['family']}"
                if docs is not None:
                    r["val_macro_f1"] = llm_val_macro_f1(rep[4:], r["family"], docs, labels)
                rows.append(r)
        else:
            if "selected" in g.columns and g["selected"].fillna(False).astype(bool).any():
                r = g[g["selected"].fillna(False).astype(bool)].iloc[0].copy(); fam = r["family"]
            else:
                fam, vf1 = val_selected_family(rep)
                if fam is None or fam not in set(g["family"]):
                    fam = g.sort_values("macro_f1").iloc[-1]["family"]
                r = g[g["family"] == fam].iloc[0].copy(); r["val_macro_f1"] = vf1
            r["pred_key"] = f"{rep}__{fam}"
            rows.append(r)
    out = pd.DataFrame(rows)
    out["system"] = out["representation"].map(pretty) + " / " + out["family"].map(fam_label)
    out["group"] = out["representation"].map(family_of)
    out["ablation"] = out["representation"].map(is_ablation)
    if "val_macro_f1" not in out.columns:
        out["val_macro_f1"] = np.nan
    out["val_macro_f1"] = pd.to_numeric(out["val_macro_f1"], errors="coerce")
    return out.reset_index(drop=True)


def representatives(sel):
    """Per family, the main-comparison system with the highest VALIDATION macro-F1; the overall
    final system is the representative with the highest validation macro-F1.  No test score is
    used for these choices."""
    m = sel[~sel["ablation"]]
    reps = []
    for grp in ["Lexical (TF-IDF)", "Frozen encoder", "Fine-tuned encoder", "LLM"]:
        g = m[(m["group"] == grp) & m["val_macro_f1"].notna()]
        if len(g):
            reps.append(g.sort_values("val_macro_f1").iloc[-1])
    R = pd.DataFrame(reps).sort_values("val_macro_f1", ascending=False).reset_index(drop=True) if reps else m.head(0)
    return R


def load_pred(key, val=False):
    f = (PREDS_VAL if val else PREDS) / f"{key}.pkl"
    return pd.read_pickle(f) if f.exists() else None


def confidence(p):
    """Confidence score for deferral: max probability when available, otherwise the
    softmax of the decision values (monotone in the top-1 margin)."""
    s = np.asarray(p["scores"], dtype=float)
    if s.min() >= 0 and np.allclose(s.sum(1), 1, atol=1e-3):
        return s.max(1)
    z = s - s.max(1, keepdims=True); e = np.exp(z); return (e / e.sum(1, keepdims=True)).max(1)


def risk_coverage(conf, correct):
    order = np.argsort(-conf); c = correct[order]
    cov = np.arange(1, len(c) + 1) / len(c); acc = np.cumsum(c) / np.arange(1, len(c) + 1)
    aurc = float(np.mean(1 - acc))
    return cov, acc, aurc


def threshold_for_target(conf_val, correct_val, target):
    """Largest coverage on validation whose accuracy on the covered subset >= target; returns the threshold."""
    order = np.argsort(-conf_val); c = correct_val[order]; acc = np.cumsum(c) / np.arange(1, len(c) + 1)
    ok = np.where(acc >= target)[0]
    if len(ok) == 0:
        return np.inf
    return float(conf_val[order][ok[-1]])


# ----------------------------------------------------------------------------- tables
def build_tables(docs, labels, df, sel):
    TAB.mkdir(parents=True, exist_ok=True)
    idx = split_indices(docs); y_test = docs["label"].values[idx["test"]]
    tables = {}
    # dataset
    st = json.loads((RESULTS / "dataset_stats.json").read_text())
    cc = pd.DataFrame({"unit": labels})
    cc["department"] = cc["unit"].map(PARENT); cc["short"] = cc["unit"].map(SHORT)
    for s in ["train", "val", "test"]:
        cc[s] = cc["unit"].map(st["split_class_counts"][s]).fillna(0).astype(int)
    cc["total"] = cc[["train", "val", "test"]].sum(axis=1); cc["share_%"] = (100 * cc["total"] / cc["total"].sum()).round(1)
    tables["dataset_classes"] = cc.sort_values("total", ascending=False)
    tables["dataset_stats"] = pd.DataFrame([{k: v for k, v in st.items() if not isinstance(v, dict)}]).T.reset_index().rename(columns={"index": "statistic", 0: "value"})
    # main + full
    cols = ["system", "group", "accuracy", "accuracy_ci", "macro_f1", "macro_f1_ci", "weighted_f1", "balanced_accuracy", "top3_accuracy",
            "lenient_accuracy", "department_accuracy", "cv_accuracy_mean", "cv_accuracy_sd", "cv_macro_f1_mean", "cv_macro_f1_sd",
            "fit_seconds_trainval", "predict_ms_per_doc", "n_features", "params"]
    cols = cols[:2] + ["val_macro_f1"] + cols[2:]
    tables["main"] = sel.loc[~sel["ablation"], [c for c in cols if c in sel.columns]].sort_values("macro_f1", ascending=False)
    tables["ablations"] = sel.loc[sel["ablation"], [c for c in cols if c in sel.columns]].sort_values(["group", "system"])
    tables["all_configurations"] = df[[c for c in cols if c in df.columns]].sort_values(["group", "system"])
    # family representatives and the final system are chosen on validation only
    R = representatives(sel)
    tables["representatives"] = R[[c for c in cols if c in R.columns]] if len(R) else R
    top = R
    pcs = []
    for _, r in top.iterrows():
        p = load_pred(sel.loc[sel["system"] == r["system"], "pred_key"].iloc[0])
        if p is None:
            continue
        t = per_class_table(p["y_true"], p["y_pred"], labels); t.insert(0, "system", r["system"]); pcs.append(t)
    if pcs:
        pc = pd.concat(pcs); pc["train_support"] = pc["unit"].map(st["split_class_counts"]["train"]).fillna(0).astype(int)
        tables["per_class"] = pc
    # McNemar among the top systems
    keys = [(r["system"], sel.loc[sel["system"] == r["system"], "pred_key"].iloc[0]) for _, r in top.iterrows()]
    preds = {s: load_pred(k) for s, k in keys}
    rows = []
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            a, b = preds[keys[i][0]], preds[keys[j][0]]
            if a is None or b is None:
                continue
            res = mcnemar_test(a["y_true"], a["y_pred"], b["y_pred"])
            rows.append({"system_a": keys[i][0], "system_b": keys[j][0], "acc_a": float((a["y_pred"] == a["y_true"]).mean()),
                         "acc_b": float((b["y_pred"] == b["y_true"]).mean()), **res})
    if rows:
        mc = pd.DataFrame(rows)
        # Holm correction
        order = np.argsort(mc["p_value"].values); m = len(mc); adj = np.empty(m)
        running = 0
        for rank, k in enumerate(order):
            running = max(running, (m - rank) * mc["p_value"].values[k]); adj[k] = min(1, running)
        mc["p_holm"] = adj; tables["mcnemar"] = mc
    # error analysis of the final system (highest validation macro-F1)
    best_sys = R.iloc[0]["system"] if len(R) else tables["main"].iloc[0]["system"]
    bp = preds.get(best_sys) if best_sys in preds else load_pred(sel.loc[sel["system"] == best_sys, "pred_key"].iloc[0])
    if bp is not None:
        yt, yp = np.asarray(bp["y_true"]), np.asarray(bp["y_pred"])
        err = yt != yp
        same_dept = np.array([PARENT[a] == PARENT[b] for a, b in zip(yt, yp)])
        ea = {"system": best_sys, "selected_on": "validation macro-F1", "n_test": len(yt), "n_errors": int(err.sum()), "errors_within_department": int((err & same_dept).sum()),
              "errors_across_departments": int((err & ~same_dept).sum()), "department_accuracy": float(same_dept.mean())}
        pairs = pd.Series([f"{SHORT[a]} -> {SHORT[b]}" for a, b in zip(yt[err], yp[err])]).value_counts().head(12)
        tables["error_summary"] = pd.DataFrame([ea]); tables["confused_pairs"] = pairs.rename_axis("true -> predicted").reset_index(name="count")
    # deferral operating points
    rows = []
    for _, r in top.iterrows():
        key = sel.loc[sel["system"] == r["system"], "pred_key"].iloc[0]
        pt, pv = load_pred(key), load_pred(key, val=True)
        if pt is None or pv is None:
            continue
        ct, cv_ = confidence(pt), confidence(pv)
        corr_t = np.asarray(pt["y_pred"]) == np.asarray(pt["y_true"]); corr_v = np.asarray(pv["y_pred"]) == np.asarray(pv["y_true"])
        cov, acc, aurc = risk_coverage(ct, corr_t)
        rec = {"system": r["system"], "AURC": aurc, "accuracy_full": float(corr_t.mean())}
        for tgt in (0.90, 0.95):
            thr = threshold_for_target(cv_, corr_v, tgt); m = ct >= thr
            rec[f"coverage@{int(tgt*100)}"] = float(m.mean()); rec[f"accuracy@{int(tgt*100)}"] = float(corr_t[m].mean()) if m.any() else np.nan
            rec[f"threshold@{int(tgt*100)}"] = thr
        rows.append(rec)
    if rows:
        tables["deferral"] = pd.DataFrame(rows)
    # LLM cascade (local -> LLM) if LLM predictions exist
    llm_keys = sorted(p.stem for p in PREDS.glob("llm_*__rag*.pkl"))
    if llm_keys and rows:
        crows = []
        for _, r in top.iterrows():
            key = sel.loc[sel["system"] == r["system"], "pred_key"].iloc[0]
            pt, pv = load_pred(key), load_pred(key, val=True)
            if pt is None or pv is None:
                continue
            for lk in llm_keys:
                from llm import final_answers
                lt = load_pred(lk)
                model, mode = lk[4:].split("__")
                lv = final_answers(model, mode, "val", labels)
                if lt is None or lv is None:
                    continue
                if len(set(pv["doc_id"]) - set(lv)) > 0:
                    continue
                fallback = docs.loc[docs["split"] != "test", "label"].value_counts().index[0]
                llm_val_pred = np.array([(lv[d] or [fallback])[0] for d in pv["doc_id"]])
                llm_test_pred = pd.Series(lt["y_pred"], index=lt["doc_id"]).reindex(pt["doc_id"]).values
                cv_, ct = confidence(pv), confidence(pt)
                yv, ytt = np.asarray(pv["y_true"]), np.asarray(pt["y_true"])
                # validation: accuracy as a function of the LLM share
                order = np.argsort(cv_); shares = np.linspace(0, 1, 101)
                def cascade_acc(share, conf, local, llm, ytrue):
                    k = int(round(share * len(conf)))
                    low = np.argsort(conf)[:k]; pred = local.copy(); pred[low] = llm[low]
                    return float((pred == ytrue).mean())
                val_curve = [cascade_acc(s, cv_, np.asarray(pv["y_pred"]), llm_val_pred, yv) for s in shares]
                llm_alone_val = float((llm_val_pred == yv).mean())
                within = [s for s, a in zip(shares, val_curve) if a >= llm_alone_val - 0.01]
                s_min = within[0] if within else 1.0
                s_best = shares[int(np.argmax(val_curve))]
                for name_, s in (("min share within 1pt of LLM", s_min), ("best val accuracy", s_best)):
                    acc_t = cascade_acc(s, ct, np.asarray(pt["y_pred"]), llm_test_pred, ytt)
                    crows.append({"local_system": r["system"], "llm": lk, "rule": name_, "llm_share": s, "test_accuracy": acc_t,
                                  "local_alone": float((pt["y_pred"] == ytt).mean()), "llm_alone": float((llm_test_pred == ytt).mean())})
        if crows:
            tables["cascade"] = pd.DataFrame(crows)
    # lexical variants and all ablations (selected classifier per representation)
    keep = ["system", "representation", "accuracy", "macro_f1", "weighted_f1", "top3_accuracy", "cv_accuracy_mean", "cv_accuracy_sd"]
    tables["ablation_lexical"] = sel[sel["representation"].str.startswith("tfidf")][keep]
    tables["ablation_other"] = sel[sel["ablation"] & ~sel["representation"].str.startswith("tfidf")][keep]
    # cost
    tables["cost"] = sel[["system", "group", "fit_seconds_trainval", "predict_ms_per_doc", "n_features"] + [c for c in ["embedding_dim", "usd_per_1000_docs"] if c in sel.columns]]
    with pd.ExcelWriter(RESULTS / "tables.xlsx") as xw:
        for k, t in tables.items():
            t.to_excel(xw, sheet_name=k[:31], index=False); t.to_csv(TAB / f"{k}.csv", index=False)
    return tables


# ----------------------------------------------------------------------------- figures
def save(fig, name):
    FIG.mkdir(exist_ok=True)
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight"); fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight"); plt.close(fig)


def fig_dataset(docs, labels):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1.1, 1.4]})
    cc = docs["label"].value_counts()
    names = [SHORT[u] for u in cc.index]
    a1.barh(names[::-1], cc.values[::-1], color=C["blue"], height=0.62)
    for i, v in enumerate(cc.values[::-1]):
        a1.text(v + 25, i, f"{v:,}", va="center", fontsize=7.5, color=C["ink2"])
    a1.set_xlabel("Documents"); a1.set_title("(a) Documents per target unit (n = %s)" % f"{len(docs):,}")
    a1.tick_params(axis="y", length=0); a1.set_xlim(0, cc.max() * 1.15)
    m = docs.set_index("date").resample("MS").size()
    a2.plot(m.index, m.values, color=C["blue"], lw=2)
    for d, lab in [("2022-01-01", "validation"), ("2022-07-01", "test")]:
        a2.axvline(pd.Timestamp(d), color=C["axis"], lw=1); a2.text(pd.Timestamp(d), m.max() * 1.02, " " + lab, fontsize=7.5, color=C["ink2"], va="bottom")
    a2.text(pd.Timestamp("2020-01-01"), m.max() * 1.02, " training", fontsize=7.5, color=C["ink2"], va="bottom")
    a2.set_ylim(0, m.max() * 1.15); a2.set_ylabel("Documents per month"); a2.set_title("(b) Monthly volume and chronological split")
    a2.yaxis.grid(True, color=C["grid"], lw=0.6); a2.set_axisbelow(True)
    import matplotlib.dates as mdates
    a2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7])); a2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    a2.tick_params(axis="x", labelsize=7.5, rotation=0)
    save(fig, "fig_dataset")


def fig_main(sel):
    d = sel.sort_values("macro_f1").reset_index(drop=True)
    h = 0.32 * len(d) + 1.2
    fig, axes = plt.subplots(1, 2, figsize=(10, h), sharey=True)
    for ax, met, title in [(axes[0], "accuracy", "(a) Accuracy"), (axes[1], "macro_f1", "(b) Macro-F1")]:
        for i, r in d.iterrows():
            lo, hi = json.loads(r[met + "_ci"]) if isinstance(r[met + "_ci"], str) else r[met + "_ci"]
            col = FAMILY_COLOR[r["group"]]
            ax.plot([lo, hi], [i, i], color=col, lw=1.5, alpha=0.8)
            ax.plot(r[met], i, "o", color=col, ms=5.5, mec="white", mew=0.8)
            if not pd.isna(r.get("cv_" + met + "_mean", np.nan)):
                ax.plot(r["cv_" + met + "_mean"], i, "|", color=C["muted"], ms=9, mew=1.2)
        ax.set_yticks(range(len(d))); ax.set_yticklabels(d["system"], fontsize=7.5)
        ax.xaxis.grid(True, color=C["grid"], lw=0.6); ax.set_axisbelow(True); ax.set_title(title); ax.tick_params(axis="y", length=0)
    handles = [plt.Line2D([], [], marker="o", color=c, ls="", label=k) for k, c in FAMILY_COLOR.items() if k in set(d["group"])]
    handles.append(plt.Line2D([], [], marker="|", color=C["muted"], ls="", label="grouped 5x2 CV mean"))
    axes[1].legend(handles=handles, loc="lower right", fontsize=7.5)
    fig.text(0.5, -0.01, "Chronological test split (second half of 2022, n = 1,653). Dots: point estimate; bars: 95% bootstrap CI.", ha="center", fontsize=7.5, color=C["ink2"])
    save(fig, "fig_main")


def fig_confusion(labels, pred, title):
    from sklearn.metrics import confusion_matrix
    order = sorted(labels, key=lambda u: (["BİDB", "SKS", "Rektörlük"].index(PARENT[u]), SHORT[u]))
    cm = confusion_matrix(pred["y_true"], pred["y_pred"], labels=order)
    norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(7.2, 6.4))
    cmap = mcolors.LinearSegmentedColormap.from_list("blue_seq", ["#ffffff"] + SEQ)
    ax.imshow(norm, cmap=cmap, vmin=0, vmax=1)
    for i in range(len(order)):
        for j in range(len(order)):
            if cm[i, j] > 0:
                ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=6.5, color="white" if norm[i, j] > 0.55 else C["ink"])
    names = [SHORT[u] for u in order]
    ax.set_xticks(range(len(order))); ax.set_xticklabels(names, rotation=45, ha="right", fontsize=7.5)
    ax.set_yticks(range(len(order))); ax.set_yticklabels([f"{n} ({cm[i].sum()})" for i, n in enumerate(names)], fontsize=7.5)
    ax.set_xlabel("Predicted unit"); ax.set_ylabel("True unit (test support)")
    # department bands
    deps = [PARENT[u] for u in order]; start = 0
    for k in range(1, len(order) + 1):
        if k == len(order) or deps[k] != deps[start]:
            ax.add_patch(plt.Rectangle((start - 0.5, start - 0.5), k - start, k - start, fill=False, ec=C["orange"], lw=1.2))
            ax.text(k - 0.45, start - 0.45, deps[start], ha="right", va="top", fontsize=7, color=C["orange"], fontweight="bold")
            start = k
    ax.set_title(title, fontsize=9)
    ax.spines[:].set_visible(False); ax.tick_params(length=0)
    save(fig, "fig_confusion")


def fig_perclass(tables, labels):
    pc = tables.get("per_class")
    if pc is None:
        return
    systems = list(dict.fromkeys(pc["system"]))[:3]
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    cols = [C["blue"], C["orange"], C["aqua"]]
    for (s, col), off in zip(zip(systems, cols), (-0.25, 0, 0.25)):
        t = pc[pc["system"] == s].set_index("unit").loc[sorted(labels, key=lambda u: -pc[(pc.system == s) & (pc.unit == u)]["train_support"].iloc[0])]
        x = np.arange(len(t)) + off
        ax.bar(x, t["f1"], width=0.25, color=col, label=s)
    t0 = pc[pc["system"] == systems[0]].set_index("unit").loc[sorted(labels, key=lambda u: -pc[(pc.system == systems[0]) & (pc.unit == u)]["train_support"].iloc[0])]
    ax.set_xticks(range(len(t0))); ax.set_xticklabels([f"{SHORT[u]}\n(n={n})" for u, n in zip(t0.index, t0["train_support"])], fontsize=6.8, rotation=35, ha="right")
    ax.set_ylabel("F1 (test)"); ax.set_ylim(0, 1); ax.yaxis.grid(True, color=C["grid"], lw=0.6); ax.set_axisbelow(True)
    ax.legend(fontsize=7, loc="upper right"); ax.set_title("Per-unit F1 ordered by training support (n = training documents)")
    save(fig, "fig_perclass")


def fig_deferral(sel, tables, labels):
    top = tables["representatives"]
    top = top[top["group"] != "LLM"] if len(top) else top
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.9), gridspec_kw={"wspace": 0.28})
    ax = axes[0]; cols = [C["blue"], C["orange"], C["aqua"], C["violet"]]
    for (_, r), col in zip(top.iterrows(), cols):
        key = sel.loc[sel["system"] == r["system"], "pred_key"].iloc[0]
        pt = load_pred(key)
        if pt is None:
            continue
        cov, acc, aurc = risk_coverage(confidence(pt), np.asarray(pt["y_pred"]) == np.asarray(pt["y_true"]))
        ax.plot(cov * 100, acc * 100, color=col, lw=2, label=f"{r['system'].replace('mean of 3 seeds', 'seed 0')} (AURC {aurc:.3f})")   # curve of one seed
    for tgt in (90, 95):
        ax.axhline(tgt, color=C["axis"], lw=0.8); ax.text(99, tgt + 0.6, f"{tgt}% target", fontsize=7, color=C["ink2"], ha="right")
    ax.set_xlabel("Coverage: documents routed automatically (%)"); ax.set_ylabel("Accuracy on routed documents (%)")
    ax.set_title("(a) Confidence-based deferral to staff"); ax.legend(fontsize=6.8, loc="lower left", bbox_to_anchor=(0.0, 0.0)); ax.set_ylim(30, 101); ax.yaxis.grid(True, color=C["grid"], lw=0.6); ax.set_axisbelow(True)
    ax.set_xlim(0, 100)
    ax2 = axes[1]
    casc = tables.get("cascade")
    if casc is not None and len(casc):
        best_local = top.iloc[0]["system"]
        key = sel.loc[sel["system"] == best_local, "pred_key"].iloc[0]; pt = load_pred(key); ct = confidence(pt)
        shares = np.linspace(0, 1, 101); all_accs = []
        for lk, col in zip(sorted(casc["llm"].unique()), cols):
            lt = load_pred(lk); llm_test = pd.Series(lt["y_pred"], index=lt["doc_id"]).reindex(pt["doc_id"]).values
            accs = []
            for s in shares:
                k = int(round(s * len(ct))); low = np.argsort(ct)[:k]; pred = np.asarray(pt["y_pred"]).copy(); pred[low] = llm_test[low]
                accs.append((pred == np.asarray(pt["y_true"])).mean() * 100)
            ax2.plot(shares * 100, accs, color=col, lw=2, label=llm_label(lk)); all_accs += accs
        lo, hi = min(all_accs), max(all_accs)
        ax2.set_ylim(lo - 0.5 * (hi - lo), hi + 0.08 * (hi - lo))      # room below the curves for the legend
        ax2.set_xlabel("Share of documents sent to the LLM (%)"); ax2.set_ylabel("Cascade accuracy (%)")
        ax2.set_title(f"(b) Cascade: {best_local.split(' / ')[0]} \u2192 LLM"); ax2.legend(fontsize=6.8, loc="lower right"); ax2.yaxis.grid(True, color=C["grid"], lw=0.6); ax2.set_axisbelow(True)
    else:
        ax2.text(0.5, 0.5, "LLM cascade: run llm.py first", ha="center", va="center", color=C["muted"]); ax2.set_axis_off()
    save(fig, "fig_deferral")


def fig_ablation(sel):
    rows = [("Preprocessing", ["tfidf_word_raw", "tfidf_word_stem", "tfidf_word_lemma", "tfidf_char_raw", "tfidf_wordchar_raw"]),
            ("Input field", ["tfidf_subject_raw", "tfidf_content_tr", "tfidf_word_raw"]),
            ("Translation (content only)", ["tfidf_content_tr", "tfidf_content_en", "emb_mpnet_multi__content_clean", "emb_mpnet_en__content_en_clean"]),
            ("Sender-unit context", [r for r in sel["representation"] if r.endswith("+sender")] + [r.replace("+sender", "") for r in sel["representation"] if r.endswith("+sender")])]
    rows = [(t, [r for r in reps if r in set(sel["representation"])]) for t, reps in rows]
    rows = [(t, reps) for t, reps in rows if reps]
    if not rows:
        return
    import textwrap
    fig, axes = plt.subplots(1, len(rows), figsize=(3.6 * len(rows) + 1, 3.4), squeeze=False)
    fig.subplots_adjust(wspace=1.1)
    for ax, (title, reps) in zip(axes[0], rows):
        d = sel.set_index("representation").loc[reps]
        y = np.arange(len(d))
        ax.barh(y + 0.18, d["accuracy"] * 100, height=0.34, color=C["blue"], label="Accuracy")
        ax.barh(y - 0.18, d["macro_f1"] * 100, height=0.34, color=C["aqua"], label="Macro-F1")
        for yy, (a, f) in zip(y, zip(d["accuracy"], d["macro_f1"])):
            ax.text(a * 100 + 0.5, yy + 0.18, f"{a*100:.1f}", va="center", fontsize=6.5, color=C["ink2"])
            ax.text(f * 100 + 0.5, yy - 0.18, f"{f*100:.1f}", va="center", fontsize=6.5, color=C["ink2"])
        ax.set_yticks(y); ax.set_yticklabels([textwrap.fill(pretty(r), 22) for r in reps], fontsize=7); ax.invert_yaxis(); ax.set_xlim(0, 100)
        ax.set_title(title); ax.tick_params(axis="y", length=0); ax.xaxis.grid(True, color=C["grid"], lw=0.6); ax.set_axisbelow(True)
    axes[0][0].legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
    save(fig, "fig_ablation")


def fig_pipeline():
    dot = r"""
digraph G {
  rankdir=TB; nodesep=0.3; ranksep=0.45; bgcolor="white"; newrank=true; compound=true;
  node [shape=box, style="rounded,filled", fontname="Helvetica", fontsize=11, color="#c3c2b7", penwidth=1, margin="0.12,0.06"];
  edge [color="#52514e", arrowsize=0.7, fontname="Helvetica", fontsize=9];
  subgraph cluster_data { label="Data layer"; fontname="Helvetica-Bold"; fontsize=11; color="#e1e0d9"; style="rounded"; rank=same; labeljust="l";
    raw [label="EDMS dispatch log\n33,828 records (2019-2022)", fillcolor="#eef4fc"];
    doc [label="Document table\n9,345 documents, 13 units\n(first dispatch = label)", fillcolor="#eef4fc"];
    split [label="Chronological partitions\ntrain / validation / test\n+ grouped 5x2 CV", fillcolor="#eef4fc"];
    raw -> doc -> split; }
  subgraph cluster_rep { label="Representation layer"; fontname="Helvetica-Bold"; fontsize=11; color="#e1e0d9"; style="rounded"; rank=same; labeljust="l";
    lex [label="Lexical\nTF-IDF word / character n-grams\n(raw, stem, lemma)", fillcolor="#e8f6f0"];
    emb [label="Frozen sentence encoders\nmultilingual and Turkish", fillcolor="#e8f6f0"];
    ft  [label="Fine-tuned encoders\nBERTurk, XLM-R", fillcolor="#e8f6f0"];
  }
  subgraph cluster_cls { label="Decision layer"; fontname="Helvetica-Bold"; fontsize=11; color="#e1e0d9"; style="rounded"; rank=same; labeljust="l";
    clf [label="Classifier grid\nLR, SVM, KNN, RF, NB, DT\n(selected on validation)", fillcolor="#fdeee8"];
    conf [label="Ranked units + confidence\n(top-1 probability)", fillcolor="#fdeee8"];
  }
  subgraph cluster_route { label="Routing layer"; fontname="Helvetica-Bold"; fontsize=11; color="#e1e0d9"; style="rounded"; rank=same; labeljust="l";
    auto [label="Automatic routing\n(top-1; top-3 as suggestion)", fillcolor="#f3f0fb"];
    llm [label="LLM second stage\n(retrieval-augmented,\nde-identified text)", fillcolor="#f3f0fb"];
    human [label="Staff review\n(top-3 suggestion)", fillcolor="#f3f0fb"];
  }
  split -> lex; split -> emb; split -> ft;
  lex -> clf; emb -> clf; clf -> conf; ft -> conf [label="softmax"];
  conf -> auto [label="confidence >= threshold"]; conf -> llm [label="low confidence"]; conf -> human [style=dashed, label="very low"];
  llm -> human [label="residual"];
}"""
    FIG.mkdir(exist_ok=True)
    (FIG / "fig_pipeline.dot").write_text(dot)
    for fmt in ("png", "pdf"):
        try:
            subprocess.run(["dot", f"-T{fmt}", "-Gdpi=250" if fmt == "png" else "-Gdpi=72", str(FIG / "fig_pipeline.dot"), "-o", str(FIG / f"fig_pipeline.{fmt}")], check=False)
        except FileNotFoundError:
            print("graphviz 'dot' not found - fig_pipeline skipped"); return


def main():
    docs, labels = load_docs()
    df = load_results(); sel = selected_systems(df, docs, labels)
    tables = build_tables(docs, labels, df, sel)
    fig_dataset(docs, labels); fig_main(sel[~sel["ablation"]]); fig_perclass(tables, labels); fig_deferral(sel, tables, labels)
    fig_ablation(sel); fig_pipeline()
    R = tables["representatives"]
    final = R.iloc[0]["system"] if len(R) else tables["main"].iloc[0]["system"]
    bp = load_pred(sel.loc[sel["system"] == final, "pred_key"].iloc[0])
    if bp is not None:
        fig_confusion(labels, bp, f"Confusion matrix on the test split: {final}")
    print(tables["main"][["system", "val_macro_f1", "accuracy", "macro_f1", "weighted_f1", "top3_accuracy", "cv_accuracy_mean"]].round(4).to_string())
    if len(R):
        print("\nfamily representatives (chosen on validation):")
        print(R[["system", "val_macro_f1", "accuracy", "macro_f1"]].round(4).to_string())
    print("\nfigures:", sorted(p.name for p in FIG.glob("*.png")))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/drive/MyDrive/ebys-routing/src/routing.py
"""
routing.py
----------
Routing mechanisms evaluated on the stored validation and test predictions of the local family
representatives (fine-tuned BERTurk: class probabilities; TF-IDF + linear SVM: decision values).

1. Calibration.  A single temperature T is fitted on the validation predictions by minimising the
   negative log-likelihood (temperature scaling; Guo et al., 2017) and applied to the test predictions.
   Expected calibration error (ECE, 15 equal-width confidence bins; Naeini et al., 2015) and reliability
   curves are reported before and after scaling.  Note: the test predictions come from the model refitted
   on train+validation, so T is transferred from the train-only model that produced the validation scores.
2. Deferral rules on the temperature-scaled confidences.  (a) empirical threshold: the largest validation
   coverage whose accuracy meets the target (as in analysis.py, which applies it to the raw confidences);
   (b) calibrated rule: route when the calibrated confidence >= target;
   (c) recency rule: the empirical threshold chosen on the last quarter of the validation half-year only.
3. Hierarchical back-off.  A document is routed to the predicted unit if the unit confidence reaches t_u;
   otherwise to the office of the predicted department (whose staff forward it inside the department) if
   the department confidence (sum of the unit probabilities of that department) reaches t_d; otherwise
   it is left to staff.  t_u and t_d are chosen on validation for the same target accuracy at their level.

    python src/routing.py        # -> results/tables/{calibration,hierarchical}.csv, routing_stats.json, figures/fig_routing.*
"""
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr

sys.path.insert(0, str(Path(__file__).resolve().parent))
from analysis import C, confidence, load_pred, plt, save, threshold_for_target   # noqa: E402  (palette, rcParams)
from common import PARENT, RESULTS, load_docs                        # noqa: E402

TAB = RESULTS / "tables"
SYSTEMS = {"BERTurk fine-tuned": ("ft_bert-base-turkish-cased__FT_s0", "prob"),
           "TF-IDF (stem) + linear SVM": ("tfidf_word_stem__LinearSVM", "margin")}
DEPTS = ["BİDB", "SKS", "Rektörlük"]
OFFICES = ["Bilgi İşlem Daire Başkanlığı", "Sağlık Kültür ve Spor Daire Başkanlığı", "Rektörlük"]   # office of each department
TARGETS = (0.90, 0.95)
RECENT_FROM = "2022-04-01"          # last quarter of the validation half-year


def logits(p, kind):
    s = np.asarray(p["scores"], dtype=float)
    return np.log(np.clip(s, 1e-12, 1.0)) if kind == "prob" else s


def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(z, y_idx):
    def nll(log_t):
        P = softmax(z / np.exp(log_t))
        return -float(np.mean(np.log(np.clip(P[np.arange(len(y_idx)), y_idx], 1e-12, 1.0))))
    r = minimize_scalar(nll, bounds=(np.log(0.02), np.log(50.0)), method="bounded")
    return float(np.exp(r.x))


def nll(P, y_idx):
    return -float(np.mean(np.log(np.clip(P[np.arange(len(y_idx)), y_idx], 1e-12, 1.0))))


def ece(conf, correct, bins=15):
    edges = np.linspace(0.0, 1.0, bins + 1); total = 0.0; curve = []
    idx = np.clip(np.digitize(conf, edges[1:-1], right=True), 0, bins - 1)
    for b in range(bins):
        m = idx == b
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
            curve.append((float(conf[m].mean()), float(correct[m].mean()), int(m.sum())))
    return float(total), curve


def dept_matrix(labels):
    return np.array([[1.0 if PARENT[l] == d else 0.0 for d in DEPTS] for l in labels])


def main():
    docs, labels = load_docs()
    lab = np.array(labels)
    date = dict(zip(docs["doc_id"].astype(int), pd.to_datetime(docs["date"])))
    M = dept_matrix(labels)
    cal_rows, hier_rows, stats, curves = [], [], {}, {}
    for name, (key, kind) in SYSTEMS.items():
        pv, pt = load_pred(key, val=True), load_pred(key)
        assert list(pv["labels"]) == labels and list(pt["labels"]) == labels
        yv, yt = np.asarray(pv["y_true"]), np.asarray(pt["y_true"])
        iv, it = np.array([labels.index(y) for y in yv]), np.array([labels.index(y) for y in yt])
        zv, zt = logits(pv, kind), logits(pt, kind)
        T = fit_temperature(zv, iv)
        Pv0, Pt0, Pv1, Pt1 = softmax(zv), softmax(zt), softmax(zv / T), softmax(zt / T)
        corr_v, corr_t = Pv0.argmax(1) == iv, Pt0.argmax(1) == it          # argmax is unchanged by T
        recent = np.array([date[int(d)] >= pd.Timestamp(RECENT_FROM) for d in pv["doc_id"]])
        row = {"system": name, "temperature": T}
        for tag, Pa, Pb in (("before", Pv0, Pt0), ("after", Pv1, Pt1)):
            row[f"ece_val_{tag}"] = ece(Pa.max(1), corr_v)[0]
            e_t, curve = ece(Pb.max(1), corr_t)
            row[f"ece_test_{tag}"] = e_t; row[f"nll_test_{tag}"] = nll(Pb, it)
            curves[(name, tag)] = curve
        T_recent = fit_temperature(zv[recent], iv[recent])
        row["temperature_recent"] = T_recent
        row["ece_test_after_recent"] = ece(softmax(zt / T_recent).max(1), corr_t)[0]
        # temperature scaling acts on the whole logit vector, so it can reorder the documents' top probabilities slightly
        row["spearman_raw_vs_scaled_test"] = float(spearmanr(confidence(pt), Pt1.max(1))[0])
        for tgt in TARGETS:
            t = int(tgt * 100)
            thr = threshold_for_target(Pv1.max(1), corr_v, tgt); m = Pt1.max(1) >= thr
            row[f"emp_cov@{t}"], row[f"emp_acc@{t}"] = float(m.mean()), float(corr_t[m].mean()) if m.any() else np.nan
            row[f"emp_val_cov@{t}"] = float((Pv1.max(1) >= thr).mean())
            raw_v = confidence(pv); row[f"raw_val_cov@{t}"] = float((raw_v >= threshold_for_target(raw_v, corr_v, tgt)).mean())
            m = Pt1.max(1) >= tgt
            row[f"cal_cov@{t}"], row[f"cal_acc@{t}"] = float(m.mean()), float(corr_t[m].mean()) if m.any() else np.nan
            thr = threshold_for_target(Pv1[recent].max(1), corr_v[recent], tgt); m = Pt1.max(1) >= thr
            row[f"rec_cov@{t}"], row[f"rec_acc@{t}"] = float(m.mean()), float(corr_t[m].mean()) if m.any() else np.nan
        cal_rows.append(row)

        # hierarchical back-off on the calibrated probabilities
        Dv, Dt = Pv1 @ M, Pt1 @ M
        dep = np.array(DEPTS)
        yv_d, yt_d = np.array([PARENT[y] for y in yv]), np.array([PARENT[y] for y in yt])
        ok_dv, ok_dt = dep[Dv.argmax(1)] == yv_d, dep[Dt.argmax(1)] == yt_d
        for tgt in TARGETS:
            t_u = threshold_for_target(Pv1.max(1), corr_v, tgt)
            rest = Pv1.max(1) < t_u
            t_d = threshold_for_target(Dv.max(1)[rest], ok_dv[rest], tgt) if rest.any() else np.inf
            r = {"system": name, "target": tgt, "t_unit": t_u, "t_dept": t_d}
            for split, P, D, corr, okd, ytrue in (("val", Pv1, Dv, corr_v, ok_dv, yv), ("test", Pt1, Dt, corr_t, ok_dt, yt)):
                unit = P.max(1) >= t_u; dpt = (~unit) & (D.max(1) >= t_d); staff = ~(unit | dpt)
                office_true = np.isin(ytrue, OFFICES)          # the office itself was the target: no forwarding needed
                r.update({f"{split}_unit_share": float(unit.mean()), f"{split}_unit_acc": float(corr[unit].mean()) if unit.any() else np.nan,
                          f"{split}_dept_share": float(dpt.mean()), f"{split}_dept_acc": float(okd[dpt].mean()) if dpt.any() else np.nan,
                          f"{split}_staff_share": float(staff.mean()), f"{split}_resolved_share": float((unit | dpt).mean()),
                          f"{split}_dept_needs_forwarding": float((dpt & okd & ~office_true).sum() / max(dpt.sum(), 1)),
                          f"{split}_dept_office_is_target": float((dpt & okd & office_true).sum() / max(dpt.sum(), 1))})
            hier_rows.append(r)
        # top-down decoding (department first, then the unit inside it) versus flat argmax
        dsel = Dt.argmax(1)
        td = np.where(M[:, dsel].T > 0, Pt1, -1.0).argmax(1)
        stats[name] = {"flat_accuracy": float(corr_t.mean()), "top_down_accuracy": float((td == it).mean()),
                       "department_accuracy_flat": float((dep[(Pt1 @ M).argmax(1)] == yt_d).mean()),
                       "department_accuracy_from_unit": float((np.array([PARENT[l] for l in lab[Pt1.argmax(1)]]) == yt_d).mean()),
                       "n_val_recent": int(recent.sum())}

    cal = pd.DataFrame(cal_rows); hier = pd.DataFrame(hier_rows)
    TAB.mkdir(parents=True, exist_ok=True)
    cal.to_csv(TAB / "calibration.csv", index=False); hier.to_csv(TAB / "hierarchical.csv", index=False)
    (TAB / "routing_stats.json").write_text(json.dumps(stats, indent=2, ensure_ascii=False))

    # ---- figure: (a) reliability of BERTurk on the test half-year; (b) where the documents go
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.0), gridspec_kw={"wspace": 0.42, "width_ratios": [1, 1.25]})
    ax = axes[0]
    ax.plot([0, 1], [0, 1], color=C["axis"], lw=1, ls="--", zorder=1)
    for tag, col, lab_ in (("before", C["blue"], "uncalibrated"), ("after", C["orange"], f"temperature-scaled (T = {cal.iloc[0]['temperature']:.2f})")):
        cv = curves[("BERTurk fine-tuned", tag)]
        x, yy, n = zip(*[c for c in cv if c[2] >= 10])
        ax.plot(x, yy, color=col, lw=1.6, marker="o", ms=4.5, mec="white", mew=1.0, label=lab_, zorder=3)
    e0, e1 = cal.iloc[0]["ece_test_before"], cal.iloc[0]["ece_test_after"]
    ax.set_xlim(0.15, 1.0); ax.set_ylim(0.15, 1.0)
    ax.set_xlabel("Confidence (mean of bin)"); ax.set_ylabel("Accuracy in bin")
    ax.set_title(f"(a) Reliability, BERTurk, test (ECE {e0:.3f} → {e1:.3f})")
    ax.legend(fontsize=7, loc="upper left"); ax.yaxis.grid(True, color=C["grid"], lw=0.6); ax.set_axisbelow(True)
    ax2 = axes[1]
    rows = []
    for sysname in SYSTEMS:
        for tgt in TARGETS:
            h = hier[(hier["system"] == sysname) & (hier["target"] == tgt)].iloc[0]
            short = "BERTurk" if sysname.startswith("BERTurk") else "TF-IDF"
            rows.append((f"{short}\n{int(tgt*100)}% target", h["test_unit_share"], h["test_dept_share"], h["test_staff_share"]))
    ylab = [r[0] for r in rows][::-1]; vals = np.array([r[1:] for r in rows])[::-1]
    left = np.zeros(len(rows))
    for j, (col, lab_) in enumerate(((C["blue"], "unit (automatic)"), (C["orange"], "department office (back-off)"), (C["violet"], "staff"))):
        ax2.barh(ylab, vals[:, j] * 100, left=left, color=col, edgecolor="white", linewidth=1.2, height=0.62, label=lab_)
        for i, v in enumerate(vals[:, j]):
            if v >= 0.07:
                ax2.text(left[i] + v * 50, i, f"{v*100:.0f}%", ha="center", va="center", fontsize=7, color="white", fontweight="bold")
        left += vals[:, j] * 100
    ax2.set_xlim(0, 100); ax2.set_xlabel("Share of test documents (%)")
    ax2.set_title("(b) Hierarchical back-off on the test half-year")
    ax2.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3)
    save(fig, "fig_routing")
    pd.set_option("display.width", 250)
    print(cal.round(4).T.to_string())
    print(hier.round(4).to_string(index=False))
    print(json.dumps(stats, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/drive/MyDrive/ebys-routing/paper/extra_stats.py
"""
extra_stats.py
--------------
Additional statistics quoted in the manuscript (written to results/tables/extra_stats.json):
rank agreement between estimates, the retrieved-neighbour vote that the RAG prompt contains,
extra McNemar tests (Holm-corrected as one family) and the cost of the cascade.
"""
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import f1_score

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from analysis import confidence, load_pred, load_results, selected_systems   # noqa: E402
from common import RESULTS, load_docs, mcnemar_test, split_indices           # noqa: E402
from llm import final_answers                                                 # noqa: E402

docs, labels = load_docs()
idx = split_indices(docs)
df = load_results()
sel = selected_systems(df, docs, labels)
out = {}

# 1. rank agreement --------------------------------------------------------------------------
loc = sel[(~sel["ablation"]) & (sel["group"] != "LLM") & sel["cv_accuracy_mean"].notna()]
rho, p = spearmanr(loc["accuracy"], loc["cv_accuracy_mean"])
out["spearman_test_vs_cv_accuracy"] = {"rho": float(rho), "p": float(p), "n": int(len(loc))}
vm = sel[(~sel["ablation"]) & sel["val_macro_f1"].notna()]
rho, p = spearmanr(vm["val_macro_f1"], vm["macro_f1"])
out["spearman_val_vs_test_macro_f1"] = {"rho": float(rho), "p": float(p), "n": int(len(vm))}
out["cv_minus_test_accuracy_points"] = {"min": float(100 * (loc["cv_accuracy_mean"] - loc["accuracy"]).min()),
                                        "max": float(100 * (loc["cv_accuracy_mean"] - loc["accuracy"]).max()),
                                        "median": float(100 * (loc["cv_accuracy_mean"] - loc["accuracy"]).median())}

# 2. vote of the ten retrieved neighbours (the labels shown in the RAG prompt) ----------------
def knn_vote(split):
    nbs = pd.read_pickle(RESULTS / "llm" / f"neighbours__rag10__{split}.pkl")
    ids = docs["doc_id"].values[idx[split]]; y = docs["label"].values[idx[split]]
    pred = []
    for d in ids:
        labs = [docs["label"].values[j] for j in nbs[int(d)]]
        counts = pd.Series(labs).value_counts()
        top = counts[counts == counts.max()].index
        pred.append(next(l for l in labs if l in top))          # tie -> label of the nearest neighbour
    pred = np.array(pred)
    return ids, y, pred
ids_t, y_t, knn_t = knn_vote("test")
_, y_v, knn_v = knn_vote("val")
out["knn10_vote"] = {"test_accuracy": float((knn_t == y_t).mean()), "test_macro_f1": float(f1_score(y_t, knn_t, labels=labels, average="macro", zero_division=0)),
                     "val_accuracy": float((knn_v == y_v).mean()), "val_macro_f1": float(f1_score(y_v, knn_v, labels=labels, average="macro", zero_division=0))}

# 3. extra McNemar tests ------------------------------------------------------------------------
def pk(contains):
    return sel.loc[sel["system"].str.contains(contains, regex=False), "pred_key"].iloc[0]
def preds(key):
    p = load_pred(key)
    return pd.Series(np.asarray(p["y_pred"]), index=p["doc_id"]).reindex(ids_t).values
P = {"BERTurk fine-tuned": preds(pk("BERTurk fine-tuned")), "Qwen3-Embedding-4B / RBF-SVM": preds(pk("Qwen3-Embedding-4B")),
     "Gemini RAG": preds("llm_gemini-3.8-flash__rag10"), "Sonnet RAG": preds("llm_claude-sonnet-5-5__rag10"),
     "GPT RAG": preds("llm_gpt-6-luna__rag10"), "TF-IDF stem / LinearSVM": preds(pk("TF-IDF word (stem)")),
     "kNN-10 vote": knn_t}

# cascade predictions (local = BERTurk fine-tuned seed 0, share chosen on validation by best accuracy)
ft_key = pk("BERTurk fine-tuned"); pt, pv = load_pred(ft_key), load_pred(ft_key, val=True)
fb = docs.loc[docs["split"] != "test", "label"].value_counts().index[0]
llm_cost = pd.read_csv(RESULTS / "results_llm.csv").set_index(["representation", "family"])
def cascade(model):
    va = final_answers(model, "rag10", "val", labels)
    llm_v = np.array([(va[d] or [fb])[0] for d in pv["doc_id"]])
    lt = load_pred(f"llm_{model}__rag10"); llm_t = pd.Series(lt["y_pred"], index=lt["doc_id"]).reindex(pt["doc_id"]).values
    cv_, ct = confidence(pv), confidence(pt)
    def run(share, conf, local, llm):
        k = int(round(share * len(conf))); low = np.argsort(conf)[:k]; p = np.asarray(local).copy(); p[low] = llm[low]; return p
    shares = np.linspace(0, 1, 101)
    accs = [(run(s, cv_, pv["y_pred"], llm_v) == np.asarray(pv["y_true"])).mean() for s in shares]
    s = float(shares[int(np.argmax(accs))])
    p_t = run(s, ct, pt["y_pred"], llm_t)
    p_t = pd.Series(p_t, index=pt["doc_id"]).reindex(ids_t).values
    usd = float(llm_cost.loc[(f"llm_{model}", "rag10"), "usd_per_1000_docs"])
    return s, p_t, usd
casc = {}
for model, name in [("gemini-3.8-flash", "Cascade BERTurk -> Gemini"), ("claude-sonnet-5-5", "Cascade BERTurk -> Sonnet")]:
    s, p_t, usd = cascade(model)
    P[name] = p_t
    casc[name] = {"llm_share": s, "test_accuracy": float((p_t == y_t).mean()),
                  "test_macro_f1": float(f1_score(y_t, p_t, labels=labels, average="macro", zero_division=0)),
                  "llm_usd_per_1000_docs_alone": usd, "llm_usd_per_1000_docs_cascade": usd * s}
out["cascade"] = casc

pairs = [("Sonnet RAG", "Gemini RAG"), ("Gemini RAG", "GPT RAG"), ("Gemini RAG", "BERTurk fine-tuned"),
         ("BERTurk fine-tuned", "Qwen3-Embedding-4B / RBF-SVM"), ("Qwen3-Embedding-4B / RBF-SVM", "TF-IDF stem / LinearSVM"),
         ("Gemini RAG", "kNN-10 vote"), ("Sonnet RAG", "kNN-10 vote"),
         ("Cascade BERTurk -> Gemini", "BERTurk fine-tuned"), ("Cascade BERTurk -> Gemini", "Gemini RAG")]
rows = []
for a, b in pairs:
    r = mcnemar_test(y_t, P[a], P[b])
    rows.append({"system_a": a, "system_b": b, "acc_a": float((P[a] == y_t).mean()), "acc_b": float((P[b] == y_t).mean()), **r})
# translation ablation on identical bodies (TF-IDF, selected classifier of each representation)
ctr, cen = load_pred(pk("TF-IDF content (TR)")), load_pred(pk("TF-IDF content (EN"))
a_ = pd.Series(ctr["y_pred"], index=ctr["doc_id"]).reindex(ids_t).values; b_ = pd.Series(cen["y_pred"], index=cen["doc_id"]).reindex(ids_t).values
rows.append({"system_a": "TF-IDF body (TR)", "system_b": "TF-IDF body (EN, translated)", "acc_a": float((a_ == y_t).mean()), "acc_b": float((b_ == y_t).mean()),
             **mcnemar_test(y_t, a_, b_)})
mc = pd.DataFrame(rows)
order = np.argsort(mc["p_value"].values); m = len(mc); adj = np.empty(m); running = 0
for rank, k in enumerate(order):
    running = max(running, (m - rank) * mc["p_value"].values[k]); adj[k] = min(1, running)
mc["p_holm"] = adj
mc.to_csv(RESULTS / "tables" / "mcnemar_extra.csv", index=False)
out["mcnemar_extra"] = mc.to_dict("records")
(RESULTS / "tables" / "extra_stats.json").write_text(json.dumps(out, indent=2, ensure_ascii=False))
print(json.dumps({k: v for k, v in out.items() if k != "mcnemar_extra"}, indent=1))
print(mc.round(4).to_string(index=False))


In [ ]:
# === 4. Yardımcılar: ön planda ve arka planda komut çalıştırma ===
FAILURES = []
BG = {}
ENV = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"}

def _quiet(line):
    l = line.lower()
    return ("warning" in l or "warn(" in l) and "retry" not in l

def run(cmd, log_name, check=True):
    print("\n$", cmd, flush=True)
    with open(f"{BASE}/logs/{log_name}.log", "a") as log:
        log.write(f"\n===== {time.ctime()} :: {cmd}\n")
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=BASE, env=ENV)
        for line in p.stdout:
            log.write(line)
            if not _quiet(line):
                print(line, end="", flush=True)
        p.wait()
    if p.returncode != 0:
        FAILURES.append(cmd)
        if check:
            raise RuntimeError(f"Komut başarısız ({p.returncode}): {cmd}")
        print(f"!! Komut başarısız ({p.returncode}), devam ediliyor: {cmd}")
        return False
    return True

def start_bg(name, cmd):
    if name in BG and BG[name][0].poll() is None:
        print(f"[arka plan] {name} zaten çalışıyor"); return
    log = open(f"{BASE}/logs/{name}.log", "a")
    log.write(f"\n===== {time.ctime()} :: {cmd}\n"); log.flush()
    p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT, cwd=BASE, env=ENV)
    BG[name] = (p, log, cmd)
    print(f"[arka plan] {name} başlatıldı -> logs/{name}.log")

def _last_line(name):
    try:
        lines = [l for l in open(f"{BASE}/logs/{name}.log", errors="ignore").read().splitlines() if l.strip() and not _quiet(l)]
        return lines[-1][:150] if lines else ""
    except Exception:
        return ""

def bg_status():
    for n, (p, _, _) in BG.items():
        st = "çalışıyor" if p.poll() is None else f"bitti (kod {p.returncode})"
        print(f"  {n:16s} {st:16s} {_last_line(n)}")

def wait_bg(every=120):
    while any(p.poll() is None for p, _, _ in BG.values()):
        print(time.strftime("%H:%M:%S"), "arka plan işleri:"); bg_status(); time.sleep(every)
    print(time.strftime("%H:%M:%S"), "tüm arka plan işleri bitti:"); bg_status()
    for n, (p, log, cmd) in BG.items():
        log.close()
        if p.returncode != 0:
            FAILURES.append(cmd)

In [ ]:
# === 5. Veri hazırlama ve Türkçe önişleme (~3 dk) ===
if not os.path.exists(f"{BASE}/data/documents.pkl"):
    run("python src/prepare_data.py", "prepare")
if not os.path.exists(f"{BASE}/data/text_variants.pkl"):
    run("python src/preprocess.py", "prepare")
st = json.load(open(f"{BASE}/results/dataset_stats.json"))
print({k: st[k] for k in ["n_dispatch_records_raw", "n_documents", "n_classes", "split_sizes"]})
ref = {"n_documents": 9345, "split_sizes": {"train": 5717, "val": 1975, "test": 1653}}
if st["n_documents"] == ref["n_documents"] and st["split_sizes"] == ref["split_sizes"]:
    print("OK: referans koşuyla birebir aynı veri seti ve bölmeler.")
else:
    print("UYARI: belge sayıları referans koşudan farklı:", ref)

In [ ]:
# === 6. TF-IDF taban çizgileri ve ablasyonlar (CPU, arka planda; ~1-1,5 saat) ===
TFIDF_CMD = ("python src/run_tfidf.py && "
             "python src/run_tfidf.py --reps tfidf_word_lemma tfidf_wordchar_raw --with_sender")
if RUN_TFIDF and PARALLEL:
    start_bg("tfidf", TFIDF_CMD)

In [ ]:
# === 7. BDM: kimliksizleştirme (GPU) + duman testi, ardından sağlayıcı başına arka planda tam koşu ===
LLM_OK = False
if RUN_LLM and LLM_MODELS:
    LLM_OK = run("python src/llm.py --smoke --models " + " ".join(LLM_MODELS) + " --modes " + " ".join(LLM_MODES), "llm", check=False)
    if LLM_OK:
        status = json.load(open(f"{BASE}/results/llm/smoke_status.json"))
        LLM_MODELS = [m for m in LLM_MODELS if status.get(m) == "ok"]
        print("Duman testini geçen BDM'ler:", LLM_MODELS)
        groups = {}
        for m in LLM_MODELS:
            groups.setdefault(_provider(m), []).append(m)
        for prov, models in groups.items():
            one = f"python src/llm.py --workers {LLM_WORKERS} --models " + " ".join(models) + " --modes " + " ".join(LLM_MODES)
            start_bg("llm_" + prov.split("_")[0].lower(), f"{one} ; {one}")   # ikinci geçiş, başarısız belgeleri yeniden dener
    else:
        print("\n!! BDM DUMAN TESTİ BAŞARISIZ. Yukarıdaki hata mesajına bakın (model adı / anahtar / kota). "
              "Diğer deneyler devam ediyor; düzelttikten sonra yalnızca 2., 4. ve 7. hücreleri yeniden çalıştırabilirsiniz.")

In [ ]:
# === 8. Gömme çıkarımı (GPU; ilk seferde ~20-40 dk, sonra önbellekten) + sınıflandırıcı ızgarası arka planda ===
GRID_CMD = "python src/run_emb_parallel.py --workers 4"   # her temsil ayrı iş ve ayrı sonuç dosyası (results_emb__*.csv)
if RUN_EMBED:
    run("python src/embed.py --models " + " ".join(EMBED_MODELS), "embed", check=False)
    run("python src/embed.py --models mpnet_multi --field content_clean", "embed", check=False)
    if os.path.exists(V1):
        run("python src/embed.py --models mpnet_en --field content_en_clean", "embed", check=False)
    if PARALLEL:
        start_bg("embed_grid", GRID_CMD)      # CPU işi; fine-tuning (GPU) ile aynı anda koşar

In [ ]:
# === 9. Uçtan uca fine-tuning: BERTurk ve XLM-R (GPU; T4 ~1-1,5 saat, L4/A100 daha kısa) ===
if RUN_FINETUNE:
    run("python src/finetune.py --cv --models " + " ".join(FINETUNE_MODELS), "finetune", check=False)
if BG: bg_status()

In [ ]:
# === 10. RAM azsa: gömme sınıflandırıcıları ve TF-IDF burada sırayla (yeterliyse zaten arka planda) ===
if RUN_EMBED and not PARALLEL:
    run(GRID_CMD, "embed_grid", check=False)
if RUN_TFIDF and not PARALLEL:
    run(TFIDF_CMD, "tfidf", check=False)
if BG: bg_status()

In [ ]:
# === 11. Arka plan işlerini bekle (TF-IDF, BDM) ve BDM sonuçlarını özetle ===
if BG:
    wait_bg()
if RUN_LLM:
    # ikinci geçiş: geçerli birim içermeyen yanıtlar ve gizli düşünmeyle üretilmiş Claude yanıtları yeniden sorulur
    run("python src/llm.py --requery --models " + " ".join(LLM_MODELS) + " --modes " + " ".join(LLM_MODES), "llm", check=False)
    run("python src/llm.py --summarise --models " + " ".join(SUMMARY_MODELS) + " --modes " + " ".join(LLM_MODES), "llm", check=False)

In [ ]:
# === 12. Analiz: tablolar (results/tables.xlsx) ve şekiller (figures/) ===
if RUN_ANALYSIS:
    run("python src/analysis.py", "analysis", check=False)
    run("python src/routing.py", "analysis", check=False)     # kalibrasyon ve hiyerarşik geri çekilme (tablolar + fig_routing)
    run("python paper/extra_stats.py", "analysis", check=False)   # sıra uyumu, kNN oylaması, ek McNemar testleri, kaskad maliyeti
    from IPython.display import Image, display
    for f in ["fig_main.png", "fig_deferral.png", "fig_routing.png"]:
        p = f"{BASE}/figures/{f}"
        if os.path.exists(p): display(Image(p, width=900))

In [ ]:
# === 13. Paketle ve indir (results + figures + logs + src; gömmeler ve kimliksizleştirilmiş metinler hariç) ===
import zipfile
stamp = time.strftime("%Y%m%d-%H%M")
out = f"/content/ebys_results_{stamp}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ["results", "figures", "logs", "src"]:
        for root, _, files in os.walk(f"{BASE}/{sub}"):
            for fn in files:
                if fn.endswith(".npy") or "masked_texts" in fn:
                    continue
                full = os.path.join(root, fn); z.write(full, os.path.relpath(full, BASE))
print("Zip:", out, f"({os.path.getsize(out)/1e6:.1f} MB)")
shutil.copy(out, f"{BASE}/ebys_results_{stamp}.zip")
print("Drive kopyası:", f"{BASE}/ebys_results_{stamp}.zip")
if FAILURES:
    print("\nUYARI - başarısız komutlar (zip yine de oluşturuldu, loglar içinde):")
    for c in FAILURES: print("  -", c)
else:
    print("\nTüm adımlar başarıyla tamamlandı.")
from google.colab import files
files.download(out)